In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 29.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import BertTokenizer
import torch as pt
import re
import numpy as np
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
from tqdm import tqdm
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import copy
import random
from collections import Counter

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, average_precision_score

from gensim.models import Word2Vec
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from nltk.stem import WordNetLemmatizer
import nltk

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# 1. Load and final procesisng of data
lemmatizer = WordNetLemmatizer()

#Lower case all text and lemmanise
def preprocess_text(text: str):
    text = text.lower()
    tokens = re.findall(r"[a-z]+", text)
    tokens = [lemmatizer.lemmatize(tok) for tok in tokens]
    return tokens

def load_texts_for_word2vec(csv_path):
    """Load texts from CSV and tokenize (lowercase + lemmatize) for Word2Vec training"""
    print(f"Loading data from {csv_path}...")
    df = pd.read_csv(csv_path)
    df['combined_text'] = df['combined_text'].astype(str)

    # Tokenize with preprocessing
    tokenized_texts = [preprocess_text(text) for text in df['combined_text']]
    labels = df['label'].values

    return tokenized_texts, labels, df

# 2. Train Word2Vec model
def train_word2vec(tokenized_texts, vector_size=100, window=5, min_count=2, workers=4, epochs=10):
    """
    Train Word2Vec model

    Parameters:
    - vector_size: dimensionality of word vectors (100, 200, 300)
    - window: context window size
    - min_count: ignores words with frequency lower than this
    - workers: number of worker threads
    - epochs: number of training epochs
    """
    print("Training Word2Vec model...")
    model = Word2Vec(
        sentences=tokenized_texts,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=workers,
        epochs=epochs,
        sg=1  # 1 for skip-gram, 0 for CBOW
    )
    print(f"Word2Vec model trained with vocabulary size: {len(model.wv)}")
    return model

# 3. Generate averaged Word2Vec features IN CHUNKS
def get_word2vec_features_chunked(tokenized_texts, w2v_model, labels, output_path, vector_size=100, chunk_size=1000):
    """Convert tokenized texts to averaged Word2Vec vectors and save in chunks"""

    # Create column names
    w2v_feature_cols = [f'w2v_{i}' for i in range(vector_size)]

    # Write header first
    header_df = pd.DataFrame(columns=w2v_feature_cols + ['label'])
    header_df.to_csv(output_path, index=False)

    # Process in chunks
    total_texts = len(tokenized_texts)
    num_chunks = (total_texts + chunk_size - 1) // chunk_size

    for chunk_idx in tqdm(range(num_chunks), desc="Processing chunks"):
        start_idx = chunk_idx * chunk_size
        end_idx = min((chunk_idx + 1) * chunk_size, total_texts)

        # Process current chunk
        chunk_features = []
        for tokens in tokenized_texts[start_idx:end_idx]:
            # Get vectors for words in vocabulary
            valid_vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]

            if valid_vectors:
                # Average the vectors
                avg_vector = np.mean(valid_vectors, axis=0)
            else:
                # Zero vector if no words found
                avg_vector = np.zeros(vector_size)

            chunk_features.append(avg_vector)

        # Create DataFrame for this chunk
        chunk_array = np.array(chunk_features)
        chunk_df = pd.DataFrame(chunk_array, columns=w2v_feature_cols)
        chunk_df['label'] = labels[start_idx:end_idx]

        # Append to CSV (without header)
        chunk_df.to_csv(output_path, mode='a', header=False, index=False)

        # Clear memory
        del chunk_features, chunk_array, chunk_df

    print(f"Word2Vec features saved to {output_path}")

# 4. Complete pipeline with chunked saving
def create_word2vec_features(csv_path, output_path, vector_size=100, window=5, chunk_size=1000):
    """Complete pipeline to create Word2Vec features with memory-efficient saving"""

    # Load and tokenize
    tokenized_texts, labels, original_df = load_texts_for_word2vec(csv_path)

    # Train Word2Vec
    w2v_model = train_word2vec(
        tokenized_texts,
        vector_size=vector_size,
        window=window
    )

    # Generate and save features in chunks
    get_word2vec_features_chunked(
        tokenized_texts,
        w2v_model,
        labels,
        output_path,
        vector_size=vector_size,
        chunk_size=chunk_size
    )

    # Save the model
    model_path = output_path.replace('.csv', '.model')
    w2v_model.save(model_path)
    print(f"Word2Vec model saved to {model_path}")

    # Return model and first few rows for verification
    verification_df = pd.read_csv(output_path, nrows=5)

    return verification_df, w2v_model

In [5]:
# w2v_df, w2v_model = create_word2vec_features(
#     csv_path='final_datasets/Combined_Train.csv',
#     output_path='preprocessed_datasets/word2vec_features.csv',
#     vector_size=100,  # 100, 200, or 300
#     window=5,
#     chunk_size=100
# )

# print("\nWord2Vec features generated!")
# print(f"Shape: {w2v_df.shape}")
# w2v_df.head()

In [6]:
def scale_data(dataframe, oversample=False):
  x = dataframe[dataframe.columns[:-1]].values
  y = dataframe[dataframe.columns[-1]].values

  scaler = StandardScaler()
  x = scaler.fit_transform(x)

  if oversample:
    ros = RandomOverSampler()
    x, y = ros.fit_resample(x, y)

  data = np.hstack((x, np.reshape(y, (-1, 1))))

  return data, x, y

In [7]:
def split_dataset(df):
    train = df.sample(frac=0.8, random_state=42)
    test = df.drop(train.index)
    print(f"Total rows in dataset: {len(df)}")
    print(f"Total rows in train set: {len(train)}")
    print(f"Total rows in test set: {len(test)}")

    print("\nClass distribution in full dataset:")
    print(df['label'].value_counts())

    print("\nClass distribution in train set:")
    print(train['label'].value_counts())

    print("\nClass distribution in test set:")
    print(test['label'].value_counts())

    train, xtrain, ytrain = scale_data(train)
    test, xtest, ytest = scale_data(test)

    print("\n KNN \n")

    return train, xtrain, ytrain, test, xtest, ytest

In [8]:
def train_ml_models(xtrain, ytrain, xtest, ytest):
    print("\n KNN \n")
    knn_model = KNeighborsClassifier(n_neighbors=5)
    knn_model.fit(xtrain, ytrain)

    ypred = knn_model.predict(xtest)
    print(classification_report(ytest, ypred))

    print("\n Gaussian NB \n")
    nb_model = GaussianNB()
    nb_model = nb_model.fit(xtrain, ytrain)

    ypred = nb_model.predict(xtest)

    print(classification_report(ytest, ypred))

    print("\n Logistica Regression \n")
    lgr_model = LogisticRegression()
    lgr_model = lgr_model.fit(xtrain, ytrain)

    ypred = lgr_model.predict(xtest)

    print(classification_report(ytest, ypred))

    print("\n Support Vector Classifier \n")
    svc_model = SVC()
    svc_model = svc_model.fit(xtrain, ytrain)

    ypred = svc_model.predict(xtest)
    print(classification_report(ytest, ypred))

    print("\n XGBoost \n")
    xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    xgb_model = xgb_model.fit(xtrain, ytrain)

    ypred = xgb_model.predict(xtest)
    print(classification_report(ytest, ypred))

    print("\n Random Forest Classifier \n")
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(xtrain, ytrain)

    ypred = rf_model.predict(xtest)
    print(classification_report(ytest, ypred))

In [9]:
class CNNModel(nn.Module):
    def __init__(self, embedding_dim, num_filters=100, filter_sizes=[3, 4, 5]):
        super(CNNModel, self).__init__()
        self.embedding_dim = embedding_dim

        # Reshape input to (batch_size, 1, embedding_dim) for conv1d
        self.conv_layers = nn.ModuleList([
            nn.Conv1d(in_channels=1, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

        # Fully connected layers
        self.fc1 = nn.Linear(num_filters * len(filter_sizes), 64)
        self.fc2 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (batch_size, embedding_dim)
        x = x.unsqueeze(1)  # (batch_size, 1, embedding_dim)

        # Apply convolutions
        conv_outputs = []
        for conv in self.conv_layers:
            conv_out = self.relu(conv(x))  # (batch_size, num_filters, length)
            pooled = torch.max(conv_out, dim=2)[0]  # Max pooling
            conv_outputs.append(pooled)

        # Concatenate all conv outputs
        x = torch.cat(conv_outputs, dim=1)  # (batch_size, num_filters * len(filter_sizes))

        x = self.dropout(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.sigmoid(self.fc2(x))

        return x

In [10]:
class LSTMModel(nn.Module):
    def __init__(self, embedding_dim, hidden_dim=128, num_layers=2):
        super(LSTMModel, self).__init__()
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_dim,
                           num_layers=num_layers, batch_first=True, dropout=0.5)
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.fc2 = nn.Linear(64, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (batch_size, embedding_dim)
        x = x.unsqueeze(1)  # (batch_size, 1, embedding_dim) - add sequence dimension

        # LSTM forward pass
        lstm_out, (hidden, cell) = self.lstm(x)  # hidden: (num_layers, batch_size, hidden_dim)

        # Use last hidden state
        x = hidden[-1]  # (batch_size, hidden_dim)

        x = self.dropout(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.sigmoid(self.fc2(x))

        return x

In [11]:
class CNNLSTMModel(nn.Module):
    def __init__(self, embedding_dim, num_filters=64, filter_sizes=[3, 4, 5],
                 hidden_dim=128, num_layers=1):
        super(CNNLSTMModel, self).__init__()
        self.embedding_dim = embedding_dim

        # CNN layers
        self.conv_layers = nn.ModuleList([
            nn.Conv1d(in_channels=1, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])

        self.relu = nn.ReLU()

        # LSTM layers - input is concatenated conv outputs
        cnn_output_dim = num_filters * len(filter_sizes)
        self.lstm = nn.LSTM(input_size=cnn_output_dim, hidden_size=hidden_dim,
                           num_layers=num_layers, batch_first=True, dropout=0.5)

        self.dropout = nn.Dropout(0.5)

        # Fully connected layers
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.fc2 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (batch_size, embedding_dim)
        x = x.unsqueeze(1)  # (batch_size, 1, embedding_dim)

        # Apply CNN
        conv_outputs = []
        for conv in self.conv_layers:
            conv_out = self.relu(conv(x))  # (batch_size, num_filters, length)
            pooled = torch.max(conv_out, dim=2)[0]  # Max pooling
            conv_outputs.append(pooled)

        x = torch.cat(conv_outputs, dim=1)  # (batch_size, cnn_output_dim)
        x = x.unsqueeze(1)  # (batch_size, 1, cnn_output_dim) - add sequence dimension

        # Apply LSTM
        lstm_out, (hidden, cell) = self.lstm(x)
        x = hidden[-1]  # (batch_size, hidden_dim)

        x = self.dropout(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.sigmoid(self.fc2(x))

        return x

In [12]:
def train_deep_learning_model(model, train_loader, test_loader, epochs=10, learning_rate=0.001, device='cpu'):
    """
    Train a PyTorch model and evaluate on test set.

    Parameters:
    - model: PyTorch model
    - train_loader: Training DataLoader
    - test_loader: Test DataLoader
    - epochs: Number of training epochs
    - learning_rate: Learning rate
    - device: 'cpu' or 'cuda'
    """

    model = model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    print(f"Training on device: {device}")
    print(f"Model: {model.__class__.__name__}")
    print("=" * 50)

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device).float()
            batch_y = batch_y.to(device).float().unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # Evaluate on test set
        model.eval()
        test_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                batch_x = batch_x.to(device).float()
                batch_y = batch_y.to(device).float().unsqueeze(1)

                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                test_loss += loss.item()

                # Calculate accuracy
                predicted = (outputs > 0.5).float()
                correct += (predicted == batch_y).sum().item()
                total += batch_y.size(0)

        avg_train_loss = train_loss / len(train_loader)
        avg_test_loss = test_loss / len(test_loader)
        accuracy = correct / total

        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f} | Accuracy: {accuracy:.4f}")

    print("=" * 50)
    print(f"{model.__class__.__name__} training completed!")

    return model

# ==================== Wrapper Function ====================
def train_all_deep_learning_models(xtrain, ytrain, xtest, ytest, epochs=10, batch_size=32):
    """
    Train CNN, LSTM, and CNN-LSTM models.

    Parameters:
    - xtrain, ytrain: Training data and labels
    - xtest, ytest: Test data and labels
    - epochs: Number of training epochs
    - batch_size: Batch size for DataLoader
    """

    # Determine device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Convert to PyTorch tensors
    xtrain_tensor = torch.from_numpy(xtrain).float()
    ytrain_tensor = torch.from_numpy(ytrain).long()
    xtest_tensor = torch.from_numpy(xtest).float()
    ytest_tensor = torch.from_numpy(ytest).long()

    # Create DataLoaders
    train_dataset = TensorDataset(xtrain_tensor, ytrain_tensor)
    test_dataset = TensorDataset(xtest_tensor, ytest_tensor)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    embedding_dim = xtrain.shape[1]

    # Train CNN
    print("\n" + "=" * 50)
    print("TRAINING CNN MODEL")
    print("=" * 50)
    cnn_model = CNNModel(embedding_dim)
    cnn_model = train_deep_learning_model(cnn_model, train_loader, test_loader, epochs=epochs, device=device)

    # Train LSTM
    print("\n" + "=" * 50)
    print("TRAINING LSTM MODEL")
    print("=" * 50)
    lstm_model = LSTMModel(embedding_dim)
    lstm_model = train_deep_learning_model(lstm_model, train_loader, test_loader, epochs=epochs, device=device)

    # Train CNN-LSTM
    print("\n" + "=" * 50)
    print("TRAINING CNN-LSTM HYBRID MODEL")
    print("=" * 50)
    cnn_lstm_model = CNNLSTMModel(embedding_dim)
    cnn_lstm_model = train_deep_learning_model(cnn_lstm_model, train_loader, test_loader, epochs=epochs, device=device)

    return cnn_model, lstm_model, cnn_lstm_model

In [13]:
def train_deep_learning_model(model, train_loader, val_loader, test_loader, epochs=10, learning_rate=0.001, device='cpu'):
    """
    Train a PyTorch model with validation set and evaluate on test set.

    Parameters:
    - model: PyTorch model
    - train_loader: Training DataLoader
    - val_loader: Validation DataLoader
    - test_loader: Test DataLoader (NOT USED during training, only for final evaluation)
    - epochs: Number of training epochs
    - learning_rate: Learning rate
    - device: 'cpu' or 'cuda'
    """

    model = model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    print(f"Training on device: {device}")
    print(f"Model: {model.__class__.__name__}")
    print("=" * 50)

    # Storage for history
    train_losses = []
    val_losses = []
    val_accuracies = []

    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0.0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device).float()
            batch_y = batch_y.to(device).float().unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # ==================== VALIDATION ====================
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(device).float()
                batch_y = batch_y.to(device).float().unsqueeze(1)

                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()

                # Calculate accuracy
                predicted = (outputs > 0.5).float()
                val_correct += (predicted == batch_y).sum().item()
                val_total += batch_y.size(0)

        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = val_correct / val_total
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)

        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Accuracy: {val_accuracy:.4f}")

    print("=" * 50)
    print(f"{model.__class__.__name__} training completed!")

    # Test-set eval
    print("\n" + "=" * 50)
    print(f"FINAL EVALUATION ON TEST SET - {model.__class__.__name__}")
    print("=" * 50)

    model.eval()
    all_predictions = []
    all_labels = []
    all_probabilities = []

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device).float()
            batch_y = batch_y.to(device).float().unsqueeze(1)

            outputs = model(batch_x)
            predicted = (outputs > 0.5).float()

            all_predictions.extend(predicted.cpu().numpy().flatten())
            all_labels.extend(batch_y.cpu().numpy().flatten())
            all_probabilities.extend(outputs.cpu().numpy().flatten())

    # Calculate metrics
    all_predictions = np.array(all_predictions)
    all_labels = np.array(all_labels)
    all_probabilities = np.array(all_probabilities)

    test_accuracy = np.mean(all_predictions == all_labels)
    test_auc = roc_auc_score(all_labels, all_probabilities)

    print(f"\nTest Accuracy: {test_accuracy:.4f}")
    print(f"Test AUC Score: {test_auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_predictions, target_names=['Fake (0)', 'Real (1)']))

    # Visualisation
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Loss curves
    axes[0, 0].plot(train_losses, label='Training Loss', marker='o')
    axes[0, 0].plot(val_losses, label='Validation Loss', marker='s')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title(f'{model.__class__.__name__} - Loss over Epochs')
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    # Accuracy curve
    axes[0, 1].plot(val_accuracies, label='Validation Accuracy', marker='o', color='green')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].set_title(f'{model.__class__.__name__} - Accuracy over Epochs')
    axes[0, 1].legend()
    axes[0, 1].grid(True)

    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_predictions)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0], cbar=False)
    axes[1, 0].set_xlabel('Predicted')
    axes[1, 0].set_ylabel('Actual')
    axes[1, 0].set_title(f'{model.__class__.__name__} - Confusion Matrix (Test Set)')
    axes[1, 0].set_xticklabels(['Fake (0)', 'Real (1)'])
    axes[1, 0].set_yticklabels(['Fake (0)', 'Real (1)'])

    # ROC Curve
    fpr, tpr, _ = roc_curve(all_labels, all_probabilities)
    axes[1, 1].plot(fpr, tpr, label=f'ROC Curve (AUC = {test_auc:.4f})', color='darkorange', lw=2)
    axes[1, 1].plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
    axes[1, 1].set_xlabel('False Positive Rate')
    axes[1, 1].set_ylabel('True Positive Rate')
    axes[1, 1].set_title(f'{model.__class__.__name__} - ROC Curve (Test Set)')
    axes[1, 1].legend()
    axes[1, 1].grid(True)

    plt.tight_layout()
    plt.show()

    return model, {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'val_accuracies': val_accuracies,
        'test_accuracy': test_accuracy,
        'test_auc': test_auc,
        'predictions': all_predictions,
        'labels': all_labels,
        'probabilities': all_probabilities
    }

In [14]:
def train_all_deep_learning_models(xtrain, ytrain, xtest, ytest, epochs=10, batch_size=32, val_split=0.2):
    """
    Train CNN, LSTM, and CNN-LSTM models with proper train/validation/test split.

    Parameters:
    - xtrain, ytrain: Training data and labels
    - xtest, ytest: Test data and labels (NOT used during training)
    - epochs: Number of training epochs
    - batch_size: Batch size for DataLoader
    - val_split: Fraction of training data to use for validation (default: 0.2)

    Verification:
    - Train set is partitioned into training and validation
    - Test set is NOT used during training, only for final evaluation
    """

    print("=" * 70)
    print("DATA SPLIT VERIFICATION")
    print("=" * 70)
    print(f"Training set size: {xtrain.shape[0]}")
    print(f"Test set size: {xtest.shape[0]}")
    print(f"Validation split: {val_split * 100}%")
    print(f"Effective training size: {int(xtrain.shape[0] * (1 - val_split))}")
    print(f"Effective validation size: {int(xtrain.shape[0] * val_split)}")
    print("✓ Test set will NOT be used during training phase")
    print("✓ Test set will ONLY be used for final evaluation")
    print("=" * 70 + "\n")

    # Determine device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Split training data into train and validation
    from sklearn.model_selection import train_test_split as sklearn_train_test_split
    xtrain_split, xval, ytrain_split, yval = sklearn_train_test_split(
        xtrain, ytrain, test_size=val_split, random_state=42, stratify=ytrain
    )

    # Convert to PyTorch tensors
    xtrain_tensor = torch.from_numpy(xtrain_split).float()
    ytrain_tensor = torch.from_numpy(ytrain_split).long()
    xval_tensor = torch.from_numpy(xval).float()
    yval_tensor = torch.from_numpy(yval).long()
    xtest_tensor = torch.from_numpy(xtest).float()
    ytest_tensor = torch.from_numpy(ytest).long()

    # Create DataLoaders
    train_dataset = TensorDataset(xtrain_tensor, ytrain_tensor)
    val_dataset = TensorDataset(xval_tensor, yval_tensor)
    test_dataset = TensorDataset(xtest_tensor, ytest_tensor)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    embedding_dim = xtrain.shape[1]

    models_history = {}

    # Train CNN
    print("\n" + "=" * 70)
    print("TRAINING CNN MODEL")
    print("=" * 70)
    cnn_model = CNNModel(embedding_dim)
    cnn_model, cnn_history = train_deep_learning_model(
        cnn_model, train_loader, val_loader, test_loader,
        epochs=epochs, device=device
    )
    models_history['CNN'] = cnn_history

    # Train LSTM
    print("\n" + "=" * 70)
    print("TRAINING LSTM MODEL")
    print("=" * 70)
    lstm_model = LSTMModel(embedding_dim)
    lstm_model, lstm_history = train_deep_learning_model(
        lstm_model, train_loader, val_loader, test_loader,
        epochs=epochs, device=device
    )
    models_history['LSTM'] = lstm_history

    # Train CNN-LSTM
    print("\n" + "=" * 70)
    print("TRAINING CNN-LSTM HYBRID MODEL")
    print("=" * 70)
    cnn_lstm_model = CNNLSTMModel(embedding_dim)
    cnn_lstm_model, cnn_lstm_history = train_deep_learning_model(
        cnn_lstm_model, train_loader, val_loader, test_loader,
        epochs=epochs, device=device
    )
    models_history['CNN-LSTM'] = cnn_lstm_history

    return cnn_model, lstm_model, cnn_lstm_model, models_history

In [15]:
def run_complete_pipeline_word2vec(
    dataset_csv_path,
    vector_size=300,
    window=5,
    chunk_size=1000,
    regenerate_features=True,
    train_ml=True,
    train_dl=True,
    dl_epochs=10,
    dl_batch_size=32,
    val_split=0.2,
    save_vectors=0,  # 0 = do not save vectors CSV, 1 = save
    w2v_output_path='word2vec_features.csv'
):
    """
    End-to-end pipeline for Word2Vec features → ML models → Deep learning models.
    - If regenerate_features is True (or file missing), trains Word2Vec and saves averaged features (when save_vectors==1).
    - Otherwise, loads the existing features CSV (when save_vectors==1).
    - When save_vectors==0, features are built in-memory and not written to disk.
    """
    print("\n" + "=" * 80)
    print("Starting Word2Vec + ML + DL pipeline")
    print("=" * 80 + "\n")

    # Step 1: Build or load Word2Vec features
    if save_vectors:
        if regenerate_features or not os.path.exists(w2v_output_path):
            print("Step 1: Training Word2Vec and creating features (saving to CSV)")
            print("-" * 80)
            _preview_df, _w2v_model = create_word2vec_features(
                csv_path=dataset_csv_path,
                output_path=w2v_output_path,
                vector_size=vector_size,
                window=window,
                chunk_size=chunk_size
            )
            feature_df = pd.read_csv(w2v_output_path)
        else:
            print("Step 1: Loading existing Word2Vec features")
            print("-" * 80)
            feature_df = pd.read_csv(w2v_output_path)
            print(f"Loaded features from {w2v_output_path}")
    else:
        print("Step 1: Training Word2Vec and creating features (in-memory, not saving CSV)")
        print("-" * 80)
        tokenized_texts, labels, _ = load_texts_for_word2vec(dataset_csv_path)
        w2v_model = train_word2vec(
            tokenized_texts,
            vector_size=vector_size,
            window=window
        )
        w2v_feature_cols = [f"w2v_{i}" for i in range(vector_size)]
        features = []
        for tokens in tokenized_texts:
            valid_vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
            if valid_vectors:
                features.append(np.mean(valid_vectors, axis=0))
            else:
                features.append(np.zeros(vector_size))
        feature_df = pd.DataFrame(np.array(features), columns=w2v_feature_cols)
        feature_df['label'] = labels
        print("Features built in memory (not saved).")

    # Step 2: Split dataset
    print("\nStep 2: Splitting dataset (80/20 train/test)")
    print("-" * 80)
    train, xtrain, ytrain, test, xtest, ytest = split_dataset(feature_df)

    # Step 3: Train ML models
    if train_ml:
        print("\nStep 3: Training traditional ML models")
        print("-" * 80)
        train_ml_models(xtrain, ytrain, xtest, ytest)
    else:
        print("\nStep 3: Skipping traditional ML models (train_ml=False)")

    # Step 4: Train deep learning models
    models_history = {}
    if train_dl:
        print("\nStep 4: Training deep learning models with validation/test split")
        print("-" * 80)
        cnn_model, lstm_model, cnn_lstm_model, models_history = train_all_deep_learning_models(
            xtrain, ytrain, xtest, ytest,
            epochs=dl_epochs,
            batch_size=dl_batch_size,
            val_split=val_split
        )
    else:
        print("\nStep 4: Skipping deep learning models (train_dl=False)")

    # Summary
    print("\n" + "=" * 80)
    print("Pipeline completed successfully")
    print("=" * 80)
    print(f"Word2Vec feature size: {xtrain.shape[1]} dimensions")
    print(f"Training samples: {len(xtrain)}")
    print(f"Test samples: {len(xtest)}")
    if train_ml:
        print("ML models trained (KNN, Naive Bayes, Logistic Regression, SVC, XGBoost, Random Forest)")
    if train_dl:
        print("DL models trained (CNN, LSTM, CNN-LSTM)")
        print(f"  - CNN test accuracy: {models_history['CNN']['test_accuracy']:.4f}")
        print(f"  - LSTM test accuracy: {models_history['LSTM']['test_accuracy']:.4f}")
        print(f"  - CNN-LSTM test accuracy: {models_history['CNN-LSTM']['test_accuracy']:.4f}")
    print("=" * 80 + "\n")

    return feature_df, xtrain, ytrain, xtest, ytest, models_history

In [16]:
# Averaged Word2Vec features for ML + sequence inputs for DL
def set_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def create_stratified_splits(df, label_col='label', test_size=0.2, val_size=0.2, seed=42):
    y = df[label_col].values
    all_indices = np.arange(len(df))

    train_idx, test_idx = train_test_split(
        all_indices,
        test_size=test_size,
        random_state=seed,
        stratify=y
    )

    y_train = y[train_idx]
    train_idx, val_idx = train_test_split(
        train_idx,
        test_size=val_size,
        random_state=seed,
        stratify=y_train
    )

    split_info = {
        'train_idx': train_idx,
        'val_idx': val_idx,
        'test_idx': test_idx
    }

    print('Split summary (stratified):')
    print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

    return split_info


def train_word2vec_model(tokenized_texts, vector_size=300, window=5, min_count=2, epochs=10, seed=42):
    model = Word2Vec(
        sentences=tokenized_texts,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=1,
        epochs=epochs,
        sg=1,
        seed=seed
    )
    return model


def average_word2vec_vector(tokens, w2v_model, vector_size):
    valid_vectors = [w2v_model.wv[tok] for tok in tokens if tok in w2v_model.wv]
    if not valid_vectors:
        return np.zeros(vector_size, dtype=np.float32)
    return np.mean(valid_vectors, axis=0).astype(np.float32)


def build_ml_matrices(tokenized_texts, labels, split_info, w2v_model, vector_size):
    features = np.array([
        average_word2vec_vector(tokens, w2v_model, vector_size)
        for tokens in tokenized_texts
    ], dtype=np.float32)

    y = np.array(labels)

    train_idx = split_info['train_idx']
    val_idx = split_info['val_idx']
    test_idx = split_info['test_idx']

    x_train = features[train_idx]
    y_train = y[train_idx]
    x_val = features[val_idx]
    y_val = y[val_idx]
    x_test = features[test_idx]
    y_test = y[test_idx]

    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_val = scaler.transform(x_val)
    x_test = scaler.transform(x_test)

    return {
        'x_train': x_train,
        'y_train': y_train,
        'x_val': x_val,
        'y_val': y_val,
        'x_test': x_test,
        'y_test': y_test,
        'scaler': scaler
    }


def evaluate_classifier(model, x, y):
    y_pred = model.predict(x)

    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(x)[:, 1]
    elif hasattr(model, 'decision_function'):
        y_score = model.decision_function(x)
    else:
        y_score = y_pred

    precision, recall, f1, _ = precision_recall_fscore_support(y, y_pred, average='binary', zero_division=0)

    return {
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc_score(y, y_score),
        'pr_auc': average_precision_score(y, y_score)
    }


def train_ml_models_v2(ml_data, seed=42):
    x_train, y_train = ml_data['x_train'], ml_data['y_train']
    x_test, y_test = ml_data['x_test'], ml_data['y_test']

    models = {
        'KNN': KNeighborsClassifier(n_neighbors=5),
        'GaussianNB': GaussianNB(),
        'LogisticRegression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=seed),
        'SVC': SVC(probability=True, class_weight='balanced', random_state=seed),
        'XGBoost': XGBClassifier(
            eval_metric='logloss',
            random_state=seed,
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6
        ),
        'RandomForest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=seed)
    }

    results = []
    fitted_models = {}

    for name, model in models.items():
        model.fit(x_train, y_train)
        fitted_models[name] = model
        metrics = evaluate_classifier(model, x_test, y_test)
        metrics['model'] = name
        results.append(metrics)

    result_df = pd.DataFrame(results).sort_values('f1', ascending=False)
    print('\nML results (test set):')
    print(result_df[['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']])

    return fitted_models, result_df


def build_vocab(train_tokenized_texts, min_freq=2):
    counter = Counter(tok for tokens in train_tokenized_texts for tok in tokens)
    vocab = {'<PAD>': 0, '<UNK>': 1}

    for token, freq in counter.items():
        if freq >= min_freq:
            vocab[token] = len(vocab)

    return vocab


def encode_tokens(tokens, vocab, max_len):
    ids = [vocab.get(tok, vocab['<UNK>']) for tok in tokens]
    ids = ids[:max_len]
    if len(ids) < max_len:
        ids.extend([vocab['<PAD>']] * (max_len - len(ids)))
    return ids


def build_embedding_matrix(vocab, w2v_model, embedding_dim):
    matrix = np.random.normal(0, 0.1, (len(vocab), embedding_dim)).astype(np.float32)
    matrix[vocab['<PAD>']] = np.zeros(embedding_dim, dtype=np.float32)

    for token, idx in vocab.items():
        if token in ('<PAD>', '<UNK>'):
            continue
        if token in w2v_model.wv:
            matrix[idx] = w2v_model.wv[token]

    return matrix


def build_sequence_dataloaders(tokenized_texts, labels, split_info, vocab, max_len=200, batch_size=32):
    y = np.array(labels, dtype=np.float32)
    x = np.array([encode_tokens(tokens, vocab, max_len) for tokens in tokenized_texts], dtype=np.int64)

    train_idx = split_info['train_idx']
    val_idx = split_info['val_idx']
    test_idx = split_info['test_idx']

    x_train, y_train = x[train_idx], y[train_idx]
    x_val, y_val = x[val_idx], y[val_idx]
    x_test, y_test = x[test_idx], y[test_idx]

    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)),
        batch_size=batch_size,
        shuffle=True
    )
    val_loader = DataLoader(
        TensorDataset(torch.from_numpy(x_val), torch.from_numpy(y_val)),
        batch_size=batch_size,
        shuffle=False
    )
    test_loader = DataLoader(
        TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test)),
        batch_size=batch_size,
        shuffle=False
    )

    return train_loader, val_loader, test_loader


class TextCNN(nn.Module):
    def __init__(self, embedding_matrix, num_filters=128, filter_sizes=(3, 4, 5), dropout=0.5):
        super().__init__()
        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))

        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters, kernel_size=k)
            for k in filter_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), 1)

    def forward(self, input_ids):
        emb = self.embedding(input_ids)
        emb = emb.transpose(1, 2)

        conv_outputs = []
        for conv in self.convs:
            out = torch.relu(conv(emb))
            pooled = torch.max(out, dim=2)[0]
            conv_outputs.append(pooled)

        x = torch.cat(conv_outputs, dim=1)
        x = self.dropout(x)
        return self.fc(x).squeeze(1)


class TextLSTM(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, num_layers=2, dropout=0.5):
        super().__init__()
        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, input_ids):
        emb = self.embedding(input_ids)
        lstm_out, _ = self.lstm(emb)
        x = lstm_out[:, -1, :]
        x = self.dropout(x)
        return self.fc(x).squeeze(1)


class TextCNNLSTM(nn.Module):
    def __init__(self, embedding_matrix, num_filters=128, kernel_size=3, hidden_dim=128, dropout=0.5):
        super().__init__()
        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))

        self.conv = nn.Conv1d(embedding_dim, num_filters, kernel_size=kernel_size, padding=kernel_size // 2)
        self.lstm = nn.LSTM(
            input_size=num_filters,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, input_ids):
        emb = self.embedding(input_ids)
        x = emb.transpose(1, 2)
        x = torch.relu(self.conv(x))
        x = x.transpose(1, 2)

        lstm_out, _ = self.lstm(x)
        x = self.dropout(lstm_out[:, -1, :])
        return self.fc(x).squeeze(1)


def _evaluate_dl(model, loader, criterion, device):
    model.eval()
    loss_sum = 0.0
    all_logits = []
    all_labels = []

    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            logits = model(batch_x)
            loss = criterion(logits, batch_y)

            loss_sum += loss.item()
            all_logits.extend(logits.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    all_logits = np.array(all_logits)
    all_labels = np.array(all_labels)
    all_probs = 1.0 / (1.0 + np.exp(-all_logits))
    all_preds = (all_probs >= 0.5).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary', zero_division=0
    )

    metrics = {
        'loss': loss_sum / max(1, len(loader)),
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc_score(all_labels, all_probs),
        'pr_auc': average_precision_score(all_labels, all_probs),
        'labels': all_labels,
        'preds': all_preds,
        'probs': all_probs
    }

    return metrics


In [17]:
def train_ml_models_cv_v2(ml_data, seed=42, cv_folds=5):
    x_train, y_train = ml_data['x_train'], ml_data['y_train']
    x_test, y_test = ml_data['x_test'], ml_data['y_test']

    models = {
        'KNN': KNeighborsClassifier(n_neighbors=5),
        'GaussianNB': GaussianNB(),
        'LogisticRegression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=seed),
        'SVC': SVC(probability=True, class_weight='balanced', random_state=seed),
        'XGBoost': XGBClassifier(
            eval_metric='logloss',
            random_state=seed,
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6
        ),
        'RandomForest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=seed)
    }

    cv_splitter = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=seed)
    scoring = {
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1',
        'roc_auc': 'roc_auc',
        'pr_auc': 'average_precision'
    }

    fitted_models = {}
    cv_rows = []
    fold_rows = []
    holdout_rows = []

    for name, model in models.items():
        cv_scores = cross_validate(
            model,
            x_train,
            y_train,
            cv=cv_splitter,
            scoring=scoring,
            return_train_score=False,
            error_score='raise'
        )

        for fold_no in range(cv_folds):
            fold_rows.append({
                'model': name,
                'fold': fold_no + 1,
                'accuracy': cv_scores['test_accuracy'][fold_no],
                'precision': cv_scores['test_precision'][fold_no],
                'recall': cv_scores['test_recall'][fold_no],
                'f1': cv_scores['test_f1'][fold_no],
                'roc_auc': cv_scores['test_roc_auc'][fold_no],
                'pr_auc': cv_scores['test_pr_auc'][fold_no]
            })

        cv_rows.append({
            'model': name,
            'cv_accuracy_mean': np.mean(cv_scores['test_accuracy']),
            'cv_accuracy_std': np.std(cv_scores['test_accuracy']),
            'cv_precision_mean': np.mean(cv_scores['test_precision']),
            'cv_precision_std': np.std(cv_scores['test_precision']),
            'cv_recall_mean': np.mean(cv_scores['test_recall']),
            'cv_recall_std': np.std(cv_scores['test_recall']),
            'cv_f1_mean': np.mean(cv_scores['test_f1']),
            'cv_f1_std': np.std(cv_scores['test_f1']),
            'cv_roc_auc_mean': np.mean(cv_scores['test_roc_auc']),
            'cv_roc_auc_std': np.std(cv_scores['test_roc_auc']),
            'cv_pr_auc_mean': np.mean(cv_scores['test_pr_auc']),
            'cv_pr_auc_std': np.std(cv_scores['test_pr_auc'])
        })

        model.fit(x_train, y_train)
        fitted_models[name] = model
        holdout_metrics = evaluate_classifier(model, x_test, y_test)
        holdout_metrics['model'] = name
        holdout_rows.append(holdout_metrics)

    cv_df = pd.DataFrame(cv_rows)
    fold_df = pd.DataFrame(fold_rows).sort_values(['model', 'fold']).reset_index(drop=True)
    holdout_df = pd.DataFrame(holdout_rows).sort_values('f1', ascending=False).reset_index(drop=True)
    holdout_prefixed = holdout_df.rename(columns={
        'accuracy': 'holdout_accuracy',
        'precision': 'holdout_precision',
        'recall': 'holdout_recall',
        'f1': 'holdout_f1',
        'roc_auc': 'holdout_roc_auc',
        'pr_auc': 'holdout_pr_auc'
    })

    ranked_df = cv_df.merge(holdout_prefixed, on='model', how='inner').sort_values(
        'holdout_f1', ascending=False
    ).reset_index(drop=True)

    print(f"\nML results with {cv_folds}-fold CV (ranked by holdout_f1):")
    print(ranked_df[[
        'model', 'holdout_f1', 'holdout_accuracy', 'cv_f1_mean', 'cv_f1_std', 'cv_roc_auc_mean', 'cv_pr_auc_mean'
    ]])

    return fitted_models, cv_df, holdout_df, ranked_df, fold_df

In [18]:
def train_dl_model_v2(model, train_loader, val_loader, test_loader, epochs=10, lr=1e-3, patience=3, device='cpu'):
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val_loss = float('inf')
    best_state = None
    best_epoch = -1
    early_stop_epoch = epochs
    wait = 0

    history = {
        'train_loss': [],
        'val_loss': [],
        'val_f1': []
    }

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss = train_loss / max(1, len(train_loader))
        val_metrics = _evaluate_dl(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_metrics['loss'])
        history['val_f1'].append(val_metrics['f1'])

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val F1: {val_metrics['f1']:.4f}"
        )

        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                early_stop_epoch = epoch + 1
                print(f"Early stopping triggered at epoch {epoch + 1}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    test_metrics = _evaluate_dl(model, test_loader, criterion, device)
    test_metrics['best_epoch'] = best_epoch
    test_metrics['early_stop_epoch'] = early_stop_epoch

    return model, history, test_metrics

def train_dl_models_v2(train_loader, val_loader, test_loader, embedding_matrix, epochs=10, lr=1e-3, patience=3):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    models = {
        'TextCNN': TextCNN(embedding_matrix),
        'TextLSTM': TextLSTM(embedding_matrix),
        'TextCNNLSTM': TextCNNLSTM(embedding_matrix)
    }

    results = []
    trained = {}

    for name, model in models.items():
        print(f"\nTraining {name} on sequence inputs...")
        trained_model, history, test_metrics = train_dl_model_v2(
            model,
            train_loader,
            val_loader,
            test_loader,
            epochs=epochs,
            lr=lr,
            patience=patience,
            device=device
        )
        trained[name] = {
            'model': trained_model,
            'history': history,
            'test_metrics': test_metrics
        }

        results.append({
            'model': name,
            'accuracy': test_metrics['accuracy'],
            'precision': test_metrics['precision'],
            'recall': test_metrics['recall'],
            'f1': test_metrics['f1'],
            'roc_auc': test_metrics['roc_auc'],
            'pr_auc': test_metrics['pr_auc'],
            'early_stop_epoch': test_metrics['early_stop_epoch']
        })

    results_df = pd.DataFrame(results).sort_values('f1', ascending=False)
    print('\nDL results (test set):')
    print(results_df)

    return trained, results_df

In [19]:
def _build_sequence_arrays(tokenized_texts, labels, vocab, max_len=200):
    y = np.array(labels, dtype=np.float32)
    x = np.array([encode_tokens(tokens, vocab, max_len) for tokens in tokenized_texts], dtype=np.int64)
    return x, y


def _build_fold_loaders_from_indices(x, y, train_idx, val_idx, batch_size=32):
    x_train, y_train = x[train_idx], y[train_idx]
    x_val, y_val = x[val_idx], y[val_idx]

    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)),
        batch_size=batch_size,
        shuffle=True
    )
    val_loader = DataLoader(
        TensorDataset(torch.from_numpy(x_val), torch.from_numpy(y_val)),
        batch_size=batch_size,
        shuffle=False
    )
    return train_loader, val_loader


def train_dl_models_cv_v2(
    tokenized_texts,
    labels,
    split_info,
    vocab,
    embedding_matrix,
    cv_folds=5,
    max_len=200,
    batch_size=32,
    epochs=10,
    lr=1e-3,
    patience=3,
    seed=42
):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    x_all, y_all = _build_sequence_arrays(tokenized_texts, labels, vocab, max_len=max_len)
    holdout_train_idx = np.concatenate([split_info['train_idx'], split_info['val_idx']]).astype(int)
    holdout_test_idx = np.array(split_info['test_idx']).astype(int)

    if len(np.intersect1d(holdout_train_idx, holdout_test_idx)) > 0:
        raise ValueError('Leakage detected: holdout train and test indices overlap.')

    y_holdout = y_all[holdout_train_idx].astype(int)
    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=seed)

    model_factories = {
        'TextCNN': lambda: TextCNN(embedding_matrix),
        'TextLSTM': lambda: TextLSTM(embedding_matrix),
        'TextCNNLSTM': lambda: TextCNNLSTM(embedding_matrix)
    }

    final_train_loader, final_val_loader, final_test_loader = build_sequence_dataloaders(
        tokenized_texts,
        labels,
        split_info,
        vocab,
        max_len=max_len,
        batch_size=batch_size
    )

    cv_rows = []
    fold_rows = []
    holdout_rows = []
    trained_models = {}

    for name, model_factory in model_factories.items():
        print(f"\n{name}: {cv_folds}-fold CV on holdout-train partition")
        model_fold_metrics = []

        for fold_no, (rel_train_idx, rel_val_idx) in enumerate(skf.split(holdout_train_idx, y_holdout), start=1):
            abs_train_idx = holdout_train_idx[rel_train_idx]
            abs_val_idx = holdout_train_idx[rel_val_idx]

            fold_train_loader, fold_val_loader = _build_fold_loaders_from_indices(
                x_all,
                y_all,
                abs_train_idx,
                abs_val_idx,
                batch_size=batch_size
            )

            fold_model = model_factory()
            _, _, fold_metrics = train_dl_model_v2(
                fold_model,
                fold_train_loader,
                fold_val_loader,
                fold_val_loader,
                epochs=epochs,
                lr=lr,
                patience=patience,
                device=device
            )

            fold_record = {
                'model': name,
                'fold': fold_no,
                'accuracy': fold_metrics['accuracy'],
                'precision': fold_metrics['precision'],
                'recall': fold_metrics['recall'],
                'f1': fold_metrics['f1'],
                'roc_auc': fold_metrics['roc_auc'],
                'pr_auc': fold_metrics['pr_auc'],
                'early_stop_epoch': fold_metrics['early_stop_epoch']
            }
            model_fold_metrics.append(fold_record)
            fold_rows.append(fold_record)

        fold_df = pd.DataFrame(model_fold_metrics)
        cv_rows.append({
            'model': name,
            'cv_accuracy_mean': fold_df['accuracy'].mean(),
            'cv_accuracy_std': fold_df['accuracy'].std(),
            'cv_precision_mean': fold_df['precision'].mean(),
            'cv_precision_std': fold_df['precision'].std(),
            'cv_recall_mean': fold_df['recall'].mean(),
            'cv_recall_std': fold_df['recall'].std(),
            'cv_f1_mean': fold_df['f1'].mean(),
            'cv_f1_std': fold_df['f1'].std(),
            'cv_roc_auc_mean': fold_df['roc_auc'].mean(),
            'cv_roc_auc_std': fold_df['roc_auc'].std(),
            'cv_pr_auc_mean': fold_df['pr_auc'].mean(),
            'cv_pr_auc_std': fold_df['pr_auc'].std()
        })

        final_model = model_factory()
        final_model, final_history, final_test_metrics = train_dl_model_v2(
            final_model,
            final_train_loader,
            final_val_loader,
            final_test_loader,
            epochs=epochs,
            lr=lr,
            patience=patience,
            device=device
        )

        trained_models[name] = {
            'model': final_model,
            'history': final_history,
            'test_metrics': final_test_metrics
        }

        holdout_rows.append({
            'model': name,
            'accuracy': final_test_metrics['accuracy'],
            'precision': final_test_metrics['precision'],
            'recall': final_test_metrics['recall'],
            'f1': final_test_metrics['f1'],
            'roc_auc': final_test_metrics['roc_auc'],
            'pr_auc': final_test_metrics['pr_auc'],
            'early_stop_epoch': final_test_metrics['early_stop_epoch']
        })

    cv_df = pd.DataFrame(cv_rows)
    holdout_df = pd.DataFrame(holdout_rows).sort_values('f1', ascending=False).reset_index(drop=True)
    fold_df = pd.DataFrame(fold_rows)

    holdout_prefixed = holdout_df.rename(columns={
        'accuracy': 'holdout_accuracy',
        'precision': 'holdout_precision',
        'recall': 'holdout_recall',
        'f1': 'holdout_f1',
        'roc_auc': 'holdout_roc_auc',
        'pr_auc': 'holdout_pr_auc',
        'early_stop_epoch': 'holdout_early_stop_epoch'
    })

    ranked_df = cv_df.merge(holdout_prefixed, on='model', how='inner').sort_values(
        'holdout_f1', ascending=False
    ).reset_index(drop=True)

    print(f"\nDL results with {cv_folds}-fold CV (ranked by holdout_f1):")
    print(ranked_df[[
        'model', 'holdout_f1', 'holdout_accuracy', 'holdout_early_stop_epoch', 'cv_f1_mean', 'cv_f1_std', 'cv_roc_auc_mean', 'cv_pr_auc_mean'
    ]])

    return trained_models, cv_df, holdout_df, ranked_df, fold_df

In [20]:
def run_hybrid_pipeline_v2(
    dataset_csv_path,
    vector_size=300,
    window=5,
    min_count=2,
    w2v_epochs=10,
    max_len=200,
    test_size=0.2,
    val_size=0.2,
    dl_epochs=10,
    dl_batch_size=32,
    dl_lr=1e-3,
    dl_patience=3,
    seed=42,
    transductive_w2v=False,
    enable_cv=True,
    cv_folds=5
):
    set_global_seed(seed)

    df = pd.read_csv(dataset_csv_path)
    df['combined_text'] = df['combined_text'].astype(str)

    tokenized_texts = [preprocess_text(text) for text in df['combined_text']]
    labels = df['label'].values

    split_info = create_stratified_splits(
        df,
        label_col='label',
        test_size=test_size,
        val_size=val_size,
        seed=seed
    )

    if transductive_w2v:
        w2v_train_tokens = tokenized_texts
        regime = 'transductive'
    else:
        w2v_train_tokens = [tokenized_texts[i] for i in split_info['train_idx']]
        regime = 'strict'

    print(f"\nWord2Vec regime: {regime}")
    w2v_model = train_word2vec_model(
        w2v_train_tokens,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        epochs=w2v_epochs,
        seed=seed
    )

    ml_data = build_ml_matrices(tokenized_texts, labels, split_info, w2v_model, vector_size)
    if enable_cv:
        ml_models, ml_results_cv, ml_results_test, ml_results, ml_fold_results = train_ml_models_cv_v2(
            ml_data,
            seed=seed,
            cv_folds=cv_folds
        )
    else:
        ml_models, ml_results = train_ml_models_v2(ml_data, seed=seed)
        ml_results_cv = None
        ml_results_test = ml_results
        ml_fold_results = None

    train_tokens = [tokenized_texts[i] for i in split_info['train_idx']]
    vocab = build_vocab(train_tokens, min_freq=min_count)
    embedding_matrix = build_embedding_matrix(vocab, w2v_model, vector_size)

    if enable_cv:
        dl_models, dl_results_cv, dl_results_test, dl_results, dl_fold_results = train_dl_models_cv_v2(
            tokenized_texts=tokenized_texts,
            labels=labels,
            split_info=split_info,
            vocab=vocab,
            embedding_matrix=embedding_matrix,
            cv_folds=cv_folds,
            max_len=max_len,
            batch_size=dl_batch_size,
            epochs=dl_epochs,
            lr=dl_lr,
            patience=dl_patience,
            seed=seed
        )
    else:
        train_loader, val_loader, test_loader = build_sequence_dataloaders(
            tokenized_texts,
            labels,
            split_info,
            vocab,
            max_len=max_len,
            batch_size=dl_batch_size
        )

        dl_models, dl_results = train_dl_models_v2(
            train_loader,
            val_loader,
            test_loader,
            embedding_matrix,
            epochs=dl_epochs,
            lr=dl_lr,
            patience=dl_patience
        )
        dl_results_cv = None
        dl_results_test = dl_results
        dl_fold_results = None

    return {
        'split_info': split_info,
        'w2v_model': w2v_model,
        'word2vec_regime': regime,
        'vector_size': vector_size,
        'max_len': max_len,
        'ml_scaler': ml_data['scaler'],
        'ml_models': ml_models,
        'ml_results': ml_results,
        'ml_results_cv': ml_results_cv,
        'ml_results_test': ml_results_test,
        'ml_fold_results': ml_fold_results,
        'dl_models': dl_models,
        'dl_results': dl_results,
        'dl_results_cv': dl_results_cv,
        'dl_results_test': dl_results_test,
        'dl_fold_results': dl_fold_results,
        'vocab': vocab
    }

In [21]:
def _prepare_cross_dataset_ml_matrix(dataset_path, w2v_model, scaler):
    df = pd.read_csv(dataset_path).dropna()
    df['combined_text'] = df['combined_text'].astype(str)
    tokenized_texts = [preprocess_text(text) for text in df['combined_text']]
    labels = np.asarray(df['label'].values, dtype=int)

    vector_size = w2v_model.vector_size
    features = np.array([
        average_word2vec_vector(tokens, w2v_model, vector_size)
        for tokens in tokenized_texts
    ], dtype=np.float32)

    x_scaled = scaler.transform(features)
    return tokenized_texts, x_scaled, labels


def _prepare_cross_dataset_dl_loader(tokenized_texts, labels, vocab, max_len=200, batch_size=64):
    x = np.array([encode_tokens(tokens, vocab, max_len) for tokens in tokenized_texts], dtype=np.int64)
    y = np.asarray(labels, dtype=np.float32)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(x), torch.from_numpy(y)),
        batch_size=batch_size,
        shuffle=False
    )
    return loader


def _build_unseen_summary_table(model_name, model_type, y_true, y_pred, metrics, target_names=('Fake (0)', 'Real (1)')):
    report = classification_report(
        y_true,
        y_pred,
        target_names=list(target_names),
        output_dict=True,
        zero_division=0
    )

    rows = []
    for class_name in target_names:
        class_report = report[class_name]
        rows.append({
            'model_type': model_type,
            'model': model_name,
            'sub_row': class_name,
            'precision': class_report['precision'],
            'recall': class_report['recall'],
            'f1': class_report['f1-score'],
            'support': class_report['support'],
            'accuracy': np.nan,
            'roc_auc': np.nan,
            'pr_auc': np.nan
        })

    weighted_report = report['weighted avg']
    rows.append({
        'model_type': model_type,
        'model': model_name,
        'sub_row': 'Weighted Avg',
        'precision': weighted_report['precision'],
        'recall': weighted_report['recall'],
        'f1': weighted_report['f1-score'],
        'support': weighted_report['support'],
        'accuracy': np.nan,
        'roc_auc': np.nan,
        'pr_auc': np.nan
    })

    rows.append({
        'model_type': model_type,
        'model': model_name,
        'sub_row': 'Overall',
        'precision': weighted_report['precision'],
        'recall': weighted_report['recall'],
        'f1': weighted_report['f1-score'],
        'support': weighted_report['support'],
        'accuracy': metrics['accuracy'],
        'roc_auc': metrics['roc_auc'],
        'pr_auc': metrics['pr_auc']
    })

    table = pd.DataFrame(rows)
    table['sub_row'] = pd.Categorical(
        table['sub_row'],
        categories=['Fake (0)', 'Real (1)', 'Weighted Avg', 'Overall'],
        ordered=True
    )
    table = table.sort_values(['model_type', 'model', 'sub_row']).set_index(['model_type', 'model', 'sub_row'])
    return table[['precision', 'recall', 'f1', 'support', 'accuracy', 'roc_auc', 'pr_auc']]


def cross_dataset_evaluation_word2vec(result_bundle, current_dataset_name, all_datasets_paths, batch_size=64):
    print("\n" + "!" * 70)
    print(f"CROSS-DATASET GENERALIZATION: {current_dataset_name}")
    print("!" * 70)

    ml_models = result_bundle['ml_models']
    dl_models = result_bundle['dl_models']
    w2v_model = result_bundle['w2v_model']
    scaler = result_bundle['ml_scaler']
    vocab = result_bundle['vocab']
    max_len = result_bundle.get('max_len', 200)

    all_results = {}

    for target_name, target_path in all_datasets_paths.items():
        if target_name == current_dataset_name:
            continue

        print("\n" + "=" * 70)
        print(f"Testing on unseen dataset: {target_name}")
        print(f"Path: {target_path}")
        print("=" * 70)

        tokenized_texts, x_ml, y_true = _prepare_cross_dataset_ml_matrix(
            target_path,
            w2v_model,
            scaler
        )
        y_true = np.asarray(y_true, dtype=int)
        print(f"Samples: {len(y_true)}")

        ml_tables = []
        for model_name, model in ml_models.items():
            y_pred = np.asarray(model.predict(x_ml), dtype=int)
            metrics = evaluate_classifier(model, x_ml, y_true)
            ml_tables.append(_build_unseen_summary_table(model_name, 'ML', y_true, y_pred, metrics))

        ml_df = pd.concat(ml_tables).sort_index()

        dl_loader = _prepare_cross_dataset_dl_loader(
            tokenized_texts,
            y_true,
            vocab,
            max_len=max_len,
            batch_size=batch_size
        )

        dl_tables = []
        criterion = nn.BCEWithLogitsLoss()

        for model_name, model_entry in dl_models.items():
            trained_model = model_entry['model'] if isinstance(model_entry, dict) else model_entry
            device = next(trained_model.parameters()).device

            dl_metrics = _evaluate_dl(trained_model, dl_loader, criterion, device)
            dl_tables.append(_build_unseen_summary_table(
                model_name,
                'DL',
                np.asarray(dl_metrics['labels'], dtype=int),
                np.asarray(dl_metrics['preds'], dtype=int),
                dl_metrics
            ))

        dl_df = pd.concat(dl_tables).sort_index()

        combined_df = pd.concat([ml_df, dl_df]).sort_index()
        print("\nCombined Unseen-Dataset Summary (ML + DL)")
        display(combined_df)

        all_results[target_name] = {
            'combined_summary': combined_df,
            'ml_summary': ml_df,
            'dl_summary': dl_df
        }

    return all_results


DATASETS = {
    'WELFake': '/content/drive/MyDrive/datasets/WELFake_processed.csv',
    'FakeNewsNet': '/content/drive/MyDrive/datasets/FakeNewsNet_processed.csv',
    'Fake_News_Detection': '/content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv',
    'ISOT': '/content/drive/MyDrive/datasets/ISOT_processed.csv',
    'Fake_News_Classification': '/content/drive/MyDrive/datasets/Fake_News_Classification_processed.csv'
}


def new_train_loop_word2vec(dataset_name, datasets_map, **pipeline_kwargs):
    source_path = datasets_map[dataset_name]
    print(f"\nInitialising Word2Vec experiment on: {dataset_name}")

    result_bundle = run_hybrid_pipeline_v2(
        dataset_csv_path=source_path,
        **pipeline_kwargs
    )

    cross_results = cross_dataset_evaluation_word2vec(
        result_bundle=result_bundle,
        current_dataset_name=dataset_name,
        all_datasets_paths=datasets_map
    )

    return result_bundle, cross_results

In [ ]:
# Running pipeline
source_dataset = 'WELFake'

result_bundle, cross_dataset_results = new_train_loop_word2vec(
    dataset_name=source_dataset,
    datasets_map=DATASETS,
    vector_size=300,
    window=5,
    min_count=2,
    w2v_epochs=10,
    max_len=200,
    test_size=0.2,
    val_size=0.2,
    dl_epochs=10,
    dl_batch_size=32,
    dl_lr=1e-3,
    dl_patience=3,
    seed=42,
    transductive_w2v=False,
    enable_cv=True,
    cv_folds=5
)

print('\nRanking criterion: holdout test F1 (primary), CV metrics as diagnostics.')

print('\nML Ranked Results (holdout-F1 primary):')
display(result_bundle['ml_results'])

if result_bundle['ml_results_cv'] is not None:
    print('\nML CV Summary:')
    display(result_bundle['ml_results_cv'].sort_values('cv_f1_mean', ascending=False))

if result_bundle['ml_results_test'] is not None:
    print('\nML Holdout Test Metrics:')
    display(result_bundle['ml_results_test'].sort_values('f1', ascending=False))

if result_bundle['ml_fold_results'] is not None:
    print('\nML Fold-level Metrics:')
    display(result_bundle['ml_fold_results'])

print('\nDL Ranked Results (holdout-F1 primary):')
display(result_bundle['dl_results'])

if result_bundle['dl_results_cv'] is not None:
    print('\nDL CV Summary:')
    display(result_bundle['dl_results_cv'].sort_values('cv_f1_mean', ascending=False))

if result_bundle['dl_results_test'] is not None:
    print('\nDL Holdout Test Metrics:')
    display(result_bundle['dl_results_test'].sort_values('f1', ascending=False))

if result_bundle['dl_fold_results'] is not None:
    print('\nDL Fold-level Metrics:')
    display(result_bundle['dl_fold_results'])

print('\nWord2Vec Regime:', result_bundle['word2vec_regime'])
print('Vocab Size:', len(result_bundle['vocab']))


Initialising Word2Vec experiment on: WELFake
Split summary (stratified):
Train: 40748 | Val: 10188 | Test: 12734

Word2Vec regime: strict

ML results with 5-fold CV (ranked by holdout_f1):
                model  holdout_f1  holdout_accuracy  cv_f1_mean  cv_f1_std  \
0                 SVC    0.944234          0.949113    0.941261   0.001805   
1  LogisticRegression    0.923130          0.929794    0.919435   0.001454   
2             XGBoost    0.922166          0.929794    0.919754   0.004206   
3        RandomForest    0.882913          0.894691    0.884884   0.003690   
4                 KNN    0.841338          0.866656    0.838727   0.003487   
5          GaussianNB    0.591236          0.705513    0.597243   0.010720   

   cv_roc_auc_mean  cv_pr_auc_mean  
0         0.987120        0.984666  
1         0.977259        0.974018  
2         0.980500        0.977438  
3         0.960958        0.952493  
4         0.932750        0.909356  
5         0.826937        0.762384  

Tex

precision    recall        f1  \
model_type model              sub_row                                       
DL         TextCNN            Fake (0)       0.214826  0.106727  0.142606   
                              Real (1)       0.752396  0.874349  0.808801   
                              Weighted Avg   0.621424  0.687328  0.646492   
                              Overall        0.621424  0.687328  0.646492   
           TextCNNLSTM        Fake (0)       0.252594  0.429914  0.318220   
                              Real (1)       0.762709  0.590243  0.665484   
                              Weighted Avg   0.638427  0.551181  0.580878   
                              Overall        0.638427  0.551181  0.580878   
           TextLSTM           Fake (0)       0.217538  0.024239  0.043618   
                              Real (1)       0.755635  0.971916  0.850237   
                              Weighted Avg   0.624535  0.741027  0.653715   
                              Overall        0.624535  0.741027  0.653715   
ML         GaussianNB         Fake (0)       0.000000  0.000000  0.000000   
                              Real (1)       0.756363  1.000000  0.861283   
                              Weighted Avg   0.572085  0.756363  0.651443   
                              Overall        0.572085  0.756363  0.651443   
           KNN                Fake (0)       0.227709  0.575348  0.326283   
                              Real (1)       0.730856  0.371444  0.492556   
                              Weighted Avg   0.608271  0.421123  0.452046   
                              Overall        0.608271  0.421123  0.452046   
           LogisticRegression Fake (0)       0.236920  0.285043  0.258763   
                              Real (1)       0.753578  0.704273  0.728092   
                              Weighted Avg   0.627701  0.602133  0.613746   
                              Overall        0.627701  0.602133  0.613746   
           RandomForest       Fake (0)       0.227780  0.126644  0.162782   
                              Real (1)       0.753879  0.861700  0.804191   
                              Weighted Avg   0.625702  0.682613  0.647920   
                              Overall        0.625702  0.682613  0.647920   
           SVC                Fake (0)       0.192771  0.003006  0.005920   
                              Real (1)       0.756169  0.995945  0.859650   
                              Weighted Avg   0.618905  0.754029  0.651651   
                              Overall        0.618905  0.754029  0.651651   
           XGBoost            Fake (0)       0.233364  0.096204  0.136243   
                              Real (1)       0.755216  0.898196  0.820524   
                              Weighted Avg   0.628074  0.702802  0.653808   
                              Overall        0.628074  0.702802  0.653808   

                                            support  accuracy   roc_auc  \
model_type model              sub_row                                     
DL         TextCNN            Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.687328  0.484739   
           TextCNNLSTM        Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.551181  0.517201   
           TextLSTM           Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.741027  0.483348   
ML         GaussianNB         Fake (0)       5322.0       NaN      


Testing on unseen dataset: Fake_News_Detection
Path: /content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv
Samples: 38650

Combined Unseen-Dataset Summary (ML + DL)


precision    recall        f1  \
model_type model              sub_row                                       
DL         TextCNN            Fake (0)       0.012740  0.012489  0.012613   
                              Real (1)       0.199684  0.202935  0.201296   
                              Weighted Avg   0.115252  0.116921  0.116079   
                              Overall        0.115252  0.116921  0.116079   
           TextCNNLSTM        Fake (0)       0.028597  0.032654  0.030491   
                              Real (1)       0.097874  0.086440  0.091802   
                              Weighted Avg   0.066585  0.062147  0.064111   
                              Overall        0.066585  0.062147  0.064111   
           TextLSTM           Fake (0)       0.011266  0.010655  0.010952   
                              Real (1)       0.219964  0.229782  0.224766   
                              Weighted Avg   0.125707  0.130815  0.128198   
                              Overall        0.125707  0.130815  0.128198   
ML         GaussianNB         Fake (0)       0.324785  0.537580  0.404928   
                              Real (1)       0.172697  0.079504  0.108882   
                              Weighted Avg   0.241386  0.286391  0.242589   
                              Overall        0.241386  0.286391  0.242589   
           KNN                Fake (0)       0.137716  0.191166  0.160098   
                              Real (1)       0.020806  0.014155  0.016848   
                              Weighted Avg   0.073608  0.094101  0.081546   
                              Overall        0.073608  0.094101  0.081546   
           LogisticRegression Fake (0)       0.041519  0.050871  0.045721   
                              Real (1)       0.040204  0.032745  0.036093   
                              Weighted Avg   0.040798  0.040931  0.040442   
                              Overall        0.040798  0.040931  0.040442   
           RandomForest       Fake (0)       0.031398  0.039184  0.034861   
                              Real (1)       0.005514  0.004388  0.004887   
                              Weighted Avg   0.017204  0.020103  0.018425   
                              Overall        0.017204  0.020103  0.018425   
           SVC                Fake (0)       0.025245  0.031107  0.027871   
                              Real (1)       0.013301  0.010758  0.011895   
                              Weighted Avg   0.018696  0.019948  0.019111   
                              Overall        0.018696  0.019948  0.019111   
           XGBoost            Fake (0)       0.027585  0.034200  0.030539   
                              Real (1)       0.008761  0.007030  0.007801   
                              Weighted Avg   0.017263  0.019301  0.018070   
                              Overall        0.017263  0.019301  0.018070   

                                            support  accuracy   roc_auc  \
model_type model              sub_row                                     
DL         TextCNN            Fake (0)      17456.0       NaN       NaN   
                              Real (1)      21194.0       NaN       NaN   
                              Weighted Avg  38650.0       NaN       NaN   
                              Overall       38650.0  0.116921  0.012073   
           TextCNNLSTM        Fake (0)      17456.0       NaN       NaN   
                              Real (1)      21194.0       NaN       NaN   
                              Weighted Avg  38650.0       NaN       NaN   
                              Overall       38650.0  0.062147  0.016760   
           TextLSTM           Fake (0)      17456.0       NaN       NaN   
                              Real (1)      21194.0       NaN       NaN   
                              Weighted Avg  38650.0       NaN       NaN   
                              Overall       38650.0  0.130815  0.016198   
ML         GaussianNB         Fake (0)      17456.0       NaN      


Testing on unseen dataset: ISOT
Path: /content/drive/MyDrive/datasets/ISOT_processed.csv
Samples: 39098

Combined Unseen-Dataset Summary (ML + DL)


...


DL Holdout Test Metrics:


,model,accuracy,precision,recall,f1,roc_auc,pr_auc,early_stop_epoch
0,TextCNN,0.962227,0.950179,0.967452,0.958737,0.993917,0.992139,5
1,TextLSTM,0.943537,0.904495,0.978878,0.940218,0.986901,0.982555,7
2,TextCNNLSTM,0.944793,0.930865,0.948753,0.939724,0.986202,0.980117,5



DL Fold-level Metrics:


,model,fold,accuracy,precision,recall,f1,roc_auc,pr_auc,early_stop_epoch
0,TextCNN,1,0.957990,0.962906,0.943735,0.953224,0.994376,0.992884,5
1,TextCNN,2,0.961323,0.949574,0.966017,0.957725,0.992855,0.990289,5
2,TextCNN,3,0.960832,0.946678,0.968189,0.957313,0.993242,0.991171,4
3,TextCNN,4,0.962698,0.936227,0.984852,0.959924,0.993637,0.991373,4
4,TextCNN,5,0.961421,0.950939,0.964726,0.957783,0.992854,0.990050,4
5,TextLSTM,1,0.951904,0.939936,0.954988,0.947402,0.987826,0.981441,6
6,TextLSTM,2,0.950427,0.930529,0.962554,0.946271,0.988542,0.985152,8
7,TextLSTM,3,0.947678,0.938438,0.946765,0.942583,0.985487,0.978059,7
8,TextLSTM,4,0.943163,0.936879,0.937892,0.937385,0.986000,0.980416,7
9,TextLSTM,5,0.944537,0.919355,0.962129,0.940256,0.985056,0.977648,6



Word2Vec Regime: strict
Vocab Size: 88564


In [ ]:
# Running pipeline
source_dataset = 'FakeNewsNet'

result_bundle, cross_dataset_results = new_train_loop_word2vec(
    dataset_name=source_dataset,
    datasets_map=DATASETS,
    vector_size=300,
    window=5,
    min_count=2,
    w2v_epochs=10,
    max_len=200,
    test_size=0.2,
    val_size=0.2,
    dl_epochs=10,
    dl_batch_size=32,
    dl_lr=1e-3,
    dl_patience=3,
    seed=42,
    transductive_w2v=False,
    enable_cv=True,
    cv_folds=5
)

print('\nRanking criterion: holdout test F1 (primary), CV metrics as diagnostics.')

print('\nML Ranked Results (holdout-F1 primary):')
display(result_bundle['ml_results'])

if result_bundle['ml_results_cv'] is not None:
    print('\nML CV Summary:')
    display(result_bundle['ml_results_cv'].sort_values('cv_f1_mean', ascending=False))

if result_bundle['ml_results_test'] is not None:
    print('\nML Holdout Test Metrics:')
    display(result_bundle['ml_results_test'].sort_values('f1', ascending=False))

if result_bundle['ml_fold_results'] is not None:
    print('\nML Fold-level Metrics:')
    display(result_bundle['ml_fold_results'])

print('\nDL Ranked Results (holdout-F1 primary):')
display(result_bundle['dl_results'])

if result_bundle['dl_results_cv'] is not None:
    print('\nDL CV Summary:')
    display(result_bundle['dl_results_cv'].sort_values('cv_f1_mean', ascending=False))

if result_bundle['dl_results_test'] is not None:
    print('\nDL Holdout Test Metrics:')
    display(result_bundle['dl_results_test'].sort_values('f1', ascending=False))

if result_bundle['dl_fold_results'] is not None:
    print('\nDL Fold-level Metrics:')
    display(result_bundle['dl_fold_results'])

print('\nWord2Vec Regime:', result_bundle['word2vec_regime'])
print('Vocab Size:', len(result_bundle['vocab']))


Initialising Word2Vec experiment on: FakeNewsNet
Split summary (stratified):
Train: 13980 | Val: 3495 | Test: 4369

Word2Vec regime: strict

ML results with 5-fold CV (ranked by holdout_f1):
                model  holdout_f1  holdout_accuracy  cv_f1_mean  cv_f1_std  \
0             XGBoost    0.892027          0.826734    0.888108   0.004399   
1        RandomForest    0.889880          0.819181    0.885302   0.001737   
2                 KNN    0.877965          0.805676    0.878245   0.005187   
3                 SVC    0.858223          0.794003    0.852154   0.010274   
4          GaussianNB    0.838482          0.759442    0.826964   0.003693   
5  LogisticRegression    0.834197          0.765621    0.839355   0.007947   

   cv_roc_auc_mean  cv_pr_auc_mean  
0         0.822617        0.925862  
1         0.818819        0.921379  
2         0.768821        0.879499  
3         0.831418        0.929393  
4         0.741484        0.874412  
5         0.825315        0.925552  

T

precision    recall        f1  \
model_type model              sub_row                                       
DL         TextCNN            Fake (0)       0.347906  0.016240  0.031032   
                              Real (1)       0.448393  0.963331  0.611948   
                              Weighted Avg   0.393486  0.445830  0.294529   
                              Overall        0.393486  0.445830  0.294529   
           TextCNNLSTM        Fake (0)       0.000000  0.000000  0.000000   
                              Real (1)       0.453589  1.000000  0.624095   
                              Weighted Avg   0.205743  0.453589  0.283083   
                              Overall        0.205743  0.453589  0.283083   
           TextLSTM           Fake (0)       0.000000  0.000000  0.000000   
                              Real (1)       0.453589  1.000000  0.624095   
                              Weighted Avg   0.205743  0.453589  0.283083   
                              Overall        0.205743  0.453589  0.283083   
ML         GaussianNB         Fake (0)       0.000000  0.000000  0.000000   
                              Real (1)       0.452005  0.993629  0.621354   
                              Weighted Avg   0.205024  0.450699  0.281839   
                              Overall        0.205024  0.450699  0.281839   
           KNN                Fake (0)       0.604224  0.088819  0.154872   
                              Real (1)       0.458638  0.929917  0.614301   
                              Weighted Avg   0.538188  0.470331  0.363264   
                              Overall        0.538188  0.470331  0.363264   
           LogisticRegression Fake (0)       0.436398  0.107387  0.172360   
                              Real (1)       0.436499  0.832929  0.572813   
                              Weighted Avg   0.436443  0.436485  0.354001   
                              Overall        0.436443  0.436485  0.354001   
           RandomForest       Fake (0)       0.613074  0.009974  0.019629   
                              Real (1)       0.454187  0.992417  0.623174   
                              Weighted Avg   0.541005  0.455599  0.293390   
                              Overall        0.541005  0.455599  0.293390   
           SVC                Fake (0)       0.525929  0.170825  0.257887   
                              Real (1)       0.449169  0.814508  0.579028   
                              Weighted Avg   0.491112  0.462793  0.403553   
                              Overall        0.491112  0.462793  0.403553   
           XGBoost            Fake (0)       0.644218  0.041794  0.078495   
                              Real (1)       0.457183  0.972195  0.621909   
                              Weighted Avg   0.559381  0.463813  0.324981   
                              Overall        0.559381  0.463813  0.324981   

                                            support  accuracy   roc_auc  \
model_type model              sub_row                                     
DL         TextCNN            Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.445830  0.428226   
           TextCNNLSTM        Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.453589  0.516385   
           TextLSTM           Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.453589  0.483892   
ML         GaussianNB         Fake (0)      34790.0       NaN      


Testing on unseen dataset: Fake_News_Detection
Path: /content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv
Samples: 38650

Combined Unseen-Dataset Summary (ML + DL)


precision    recall        f1  \
model_type model              sub_row                                       
DL         TextCNN            Fake (0)       0.608863  0.036205  0.068346   
                              Real (1)       0.552696  0.980844  0.707003   
                              Weighted Avg   0.578063  0.554204  0.418558   
                              Overall        0.578063  0.554204  0.418558   
           TextCNNLSTM        Fake (0)       0.000000  0.000000  0.000000   
                              Real (1)       0.548357  1.000000  0.708308   
                              Weighted Avg   0.300695  0.548357  0.388406   
                              Overall        0.300695  0.548357  0.388406   
           TextLSTM           Fake (0)       0.000000  0.000000  0.000000   
                              Real (1)       0.548357  1.000000  0.708308   
                              Weighted Avg   0.300695  0.548357  0.388406   
                              Overall        0.300695  0.548357  0.388406   
ML         GaussianNB         Fake (0)       0.000000  0.000000  0.000000   
                              Real (1)       0.548357  1.000000  0.708308   
                              Weighted Avg   0.300695  0.548357  0.388406   
                              Overall        0.300695  0.548357  0.388406   
           KNN                Fake (0)       0.372485  0.084842  0.138205   
                              Real (1)       0.539280  0.882278  0.669399   
                              Weighted Avg   0.463948  0.522122  0.429489   
                              Overall        0.463948  0.522122  0.429489   
           LogisticRegression Fake (0)       0.540491  0.186583  0.277404   
                              Real (1)       0.564768  0.869350  0.684715   
                              Weighted Avg   0.553804  0.560983  0.500756   
                              Overall        0.553804  0.560983  0.500756   
           RandomForest       Fake (0)       0.183024  0.003953  0.007738   
                              Real (1)       0.545711  0.985468  0.702440   
                              Weighted Avg   0.381906  0.542173  0.388683   
                              Overall        0.381906  0.542173  0.388683   
           SVC                Fake (0)       0.471896  0.216430  0.296756   
                              Real (1)       0.553648  0.800510  0.654578   
                              Weighted Avg   0.516726  0.536714  0.492970   
                              Overall        0.516726  0.536714  0.492970   
           XGBoost            Fake (0)       0.313380  0.030591  0.055741   
                              Real (1)       0.541980  0.944796  0.688820   
                              Weighted Avg   0.438735  0.531902  0.402894   
                              Overall        0.438735  0.531902  0.402894   

                                            support  accuracy   roc_auc  \
model_type model              sub_row                                     
DL         TextCNN            Fake (0)      17456.0       NaN       NaN   
                              Real (1)      21194.0       NaN       NaN   
                              Weighted Avg  38650.0       NaN       NaN   
                              Overall       38650.0  0.554204  0.573912   
           TextCNNLSTM        Fake (0)      17456.0       NaN       NaN   
                              Real (1)      21194.0       NaN       NaN   
                              Weighted Avg  38650.0       NaN       NaN   
                              Overall       38650.0  0.548357  0.392516   
           TextLSTM           Fake (0)      17456.0       NaN       NaN   
                              Real (1)      21194.0       NaN       NaN   
                              Weighted Avg  38650.0       NaN       NaN   
                              Overall       38650.0  0.548357  0.528225   
ML         GaussianNB         Fake (0)      17456.0       NaN      


Testing on unseen dataset: ISOT
Path: /content/drive/MyDrive/datasets/ISOT_processed.csv
Samples: 39098

Combined Unseen-Dataset Summary (ML + DL)


...


DL Holdout Test Metrics:


,model,accuracy,precision,recall,f1,roc_auc,pr_auc,early_stop_epoch
0,TextCNN,0.837720,0.858168,0.940998,0.897676,0.860249,0.942111,5
1,TextLSTM,0.756466,0.756466,1.000000,0.861350,0.501426,0.756992,10
2,TextCNNLSTM,0.756466,0.756466,1.000000,0.861350,0.500000,0.756466,5



DL Fold-level Metrics:


,model,fold,accuracy,precision,recall,f1,roc_auc,pr_auc,early_stop_epoch
0,TextCNN,1,0.832618,0.851485,0.943268,0.895030,0.857310,0.940168,4
1,TextCNN,2,0.835479,0.869069,0.921331,0.894437,0.851075,0.939667,4
2,TextCNN,3,0.833190,0.875091,0.909194,0.891817,0.856550,0.939206,5
3,TextCNN,4,0.830329,0.836728,0.963678,0.895727,0.850383,0.939632,4
4,TextCNN,5,0.836624,0.854795,0.944381,0.897358,0.846706,0.935676,4
5,TextLSTM,1,0.756509,0.756509,1.000000,0.861378,0.500000,0.756509,9
6,TextLSTM,2,0.756509,0.756509,1.000000,0.861378,0.500000,0.756509,10
7,TextLSTM,3,0.756223,0.756223,1.000000,0.861193,0.500000,0.756223,5
8,TextLSTM,4,0.756223,0.756223,1.000000,0.861193,0.499394,0.756046,4
9,TextLSTM,5,0.756223,0.756223,1.000000,0.861193,0.497386,0.755485,7



Word2Vec Regime: strict
Vocab Size: 7624


In [ ]:
# Running pipeline
source_dataset = 'Fake_News_Detection'

result_bundle, cross_dataset_results = new_train_loop_word2vec(
    dataset_name=source_dataset,
    datasets_map=DATASETS,
    vector_size=300,
    window=5,
    min_count=2,
    w2v_epochs=10,
    max_len=200,
    test_size=0.2,
    val_size=0.2,
    dl_epochs=10,
    dl_batch_size=32,
    dl_lr=1e-3,
    dl_patience=3,
    seed=42,
    transductive_w2v=False,
    enable_cv=True,
    cv_folds=5
)

print('\nRanking criterion: holdout test F1 (primary), CV metrics as diagnostics.')

print('\nML Ranked Results (holdout-F1 primary):')
display(result_bundle['ml_results'])

if result_bundle['ml_results_cv'] is not None:
    print('\nML CV Summary:')
    display(result_bundle['ml_results_cv'].sort_values('cv_f1_mean', ascending=False))

if result_bundle['ml_results_test'] is not None:
    print('\nML Holdout Test Metrics:')
    display(result_bundle['ml_results_test'].sort_values('f1', ascending=False))

if result_bundle['ml_fold_results'] is not None:
    print('\nML Fold-level Metrics:')
    display(result_bundle['ml_fold_results'])

print('\nDL Ranked Results (holdout-F1 primary):')
display(result_bundle['dl_results'])

if result_bundle['dl_results_cv'] is not None:
    print('\nDL CV Summary:')
    display(result_bundle['dl_results_cv'].sort_values('cv_f1_mean', ascending=False))

if result_bundle['dl_results_test'] is not None:
    print('\nDL Holdout Test Metrics:')
    display(result_bundle['dl_results_test'].sort_values('f1', ascending=False))

if result_bundle['dl_fold_results'] is not None:
    print('\nDL Fold-level Metrics:')
    display(result_bundle['dl_fold_results'])

print('\nWord2Vec Regime:', result_bundle['word2vec_regime'])
print('Vocab Size:', len(result_bundle['vocab']))


Initialising Word2Vec experiment on: Fake_News_Detection
Split summary (stratified):
Train: 24736 | Val: 6184 | Test: 7730

Word2Vec regime: strict

ML results with 5-fold CV (ranked by holdout_f1):
                model  holdout_f1  holdout_accuracy  cv_f1_mean  cv_f1_std  \
0                 SVC    0.986501          0.985123    0.986875   0.001875   
1  LogisticRegression    0.985039          0.983571    0.983926   0.001197   
2             XGBoost    0.976897          0.974515    0.975824   0.002181   
3        RandomForest    0.957410          0.952523    0.956961   0.002500   
4                 KNN    0.952260          0.946572    0.954444   0.001548   
5          GaussianNB    0.906571          0.898836    0.906773   0.005680   

   cv_roc_auc_mean  cv_pr_auc_mean  
0         0.998493        0.998488  
1         0.997914        0.997930  
2         0.996489        0.996884  
3         0.989366        0.990297  
4         0.982622        0.976010  
5         0.952526        0.940

precision    recall        f1  \
model_type model              sub_row                                       
DL         TextCNN            Fake (0)       0.240717  0.245042  0.242860   
                              Real (1)       0.070430  0.068906  0.069660   
                              Weighted Avg   0.163477  0.165148  0.164298   
                              Overall        0.163477  0.165148  0.164298   
           TextCNNLSTM        Fake (0)       0.245779  0.251911  0.248808   
                              Real (1)       0.070898  0.068767  0.069816   
                              Weighted Avg   0.166455  0.168839  0.167619   
                              Overall        0.166455  0.168839  0.167619   
           TextLSTM           Fake (0)       0.243509  0.244524  0.244016   
                              Real (1)       0.085331  0.084903  0.085117   
                              Weighted Avg   0.171762  0.172122  0.171941   
                              Overall        0.171762  0.172122  0.171941   
ML         GaussianNB         Fake (0)       0.331885  0.335815  0.333838   
                              Real (1)       0.188317  0.185630  0.186964   
                              Weighted Avg   0.266764  0.267693  0.267218   
                              Overall        0.266764  0.267693  0.267218   
           KNN                Fake (0)       0.232383  0.223139  0.227667   
                              Real (1)       0.106959  0.112084  0.109462   
                              Weighted Avg   0.175492  0.172766  0.174051   
                              Overall        0.175492  0.172766  0.174051   
           LogisticRegression Fake (0)       0.231363  0.231590  0.231476   
                              Real (1)       0.073251  0.073165  0.073208   
                              Weighted Avg   0.159645  0.159730  0.159688   
                              Overall        0.159645  0.159730  0.159688   
           RandomForest       Fake (0)       0.240636  0.236361  0.238479   
                              Real (1)       0.099363  0.101489  0.100415   
                              Weighted Avg   0.176556  0.175185  0.175855   
                              Overall        0.176556  0.175185  0.175855   
           SVC                Fake (0)       0.228628  0.224231  0.226408   
                              Real (1)       0.086636  0.088643  0.087628   
                              Weighted Avg   0.164222  0.162730  0.163459   
                              Overall        0.164222  0.162730  0.163459   
           XGBoost            Fake (0)       0.221714  0.220293  0.221001   
                              Real (1)       0.067931  0.068456  0.068192   
                              Weighted Avg   0.151960  0.151421  0.151689   
                              Overall        0.151960  0.151421  0.151689   

                                            support  accuracy   roc_auc  \
model_type model              sub_row                                     
DL         TextCNN            Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.165148  0.064114   
           TextCNNLSTM        Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.168839  0.104989   
           TextLSTM           Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.172122  0.097629   
ML         GaussianNB         Fake (0)      34790.0       NaN      


Testing on unseen dataset: FakeNewsNet
Path: /content/drive/MyDrive/datasets/FakeNewsNet_processed.csv
Samples: 21844

Combined Unseen-Dataset Summary (ML + DL)


precision    recall        f1  \
model_type model              sub_row                                       
DL         TextCNN            Fake (0)       0.245691  0.889139  0.384997   
                              Real (1)       0.771672  0.120688  0.208730   
                              Weighted Avg   0.643523  0.307911  0.251675   
                              Overall        0.643523  0.307911  0.251675   
           TextCNNLSTM        Fake (0)       0.240016  0.680947  0.354929   
                              Real (1)       0.748258  0.305471  0.433833   
                              Weighted Avg   0.624432  0.396951  0.414609   
                              Overall        0.624432  0.396951  0.414609   
           TextLSTM           Fake (0)       0.247211  0.466366  0.323135   
                              Real (1)       0.759404  0.542549  0.632917   
                              Weighted Avg   0.634615  0.523988  0.557443   
                              Overall        0.634615  0.523988  0.557443   
ML         GaussianNB         Fake (0)       0.193887  0.076287  0.109493   
                              Real (1)       0.751089  0.897833  0.817931   
                              Weighted Avg   0.615334  0.697674  0.645330   
                              Overall        0.615334  0.697674  0.645330   
           KNN                Fake (0)       0.241989  0.828636  0.374586   
                              Real (1)       0.748066  0.163903  0.268891   
                              Weighted Avg   0.624767  0.325856  0.294642   
                              Overall        0.624767  0.325856  0.294642   
           LogisticRegression Fake (0)       0.239554  0.686208  0.355132   
                              Real (1)       0.746931  0.298330  0.426366   
                              Weighted Avg   0.623316  0.392831  0.409011   
                              Overall        0.623316  0.392831  0.409011   
           RandomForest       Fake (0)       0.245873  0.763998  0.372021   
                              Real (1)       0.763331  0.245188  0.371158   
                              Weighted Avg   0.637260  0.371589  0.371368   
                              Overall        0.637260  0.371589  0.371368   
           SVC                Fake (0)       0.224980  0.313416  0.261935   
                              Real (1)       0.746778  0.652221  0.696304   
                              Weighted Avg   0.619648  0.569676  0.590476   
                              Overall        0.619648  0.569676  0.590476   
           XGBoost            Fake (0)       0.245556  0.882563  0.384213   
                              Real (1)       0.769882  0.126559  0.217382   
                              Weighted Avg   0.642137  0.310749  0.258028   
                              Overall        0.642137  0.310749  0.258028   

                                            support  accuracy   roc_auc  \
model_type model              sub_row                                     
DL         TextCNN            Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.307911  0.507323   
           TextCNNLSTM        Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.396951  0.485349   
           TextLSTM           Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.523988  0.503370   
ML         GaussianNB         Fake (0)       5322.0       NaN      


Testing on unseen dataset: ISOT
Path: /content/drive/MyDrive/datasets/ISOT_processed.csv
Samples: 39098

Combined Unseen-Dataset Summary (ML + DL)


...


DL Holdout Test Metrics:


,model,accuracy,precision,recall,f1,roc_auc,pr_auc,early_stop_epoch
0,TextCNN,0.988745,0.988241,0.991272,0.989754,0.999255,0.999363,4
1,TextLSTM,0.983182,0.980585,0.988912,0.984731,0.997198,0.995947,6
2,TextCNNLSTM,0.980854,0.978703,0.986553,0.982613,0.996634,0.996913,9



DL Fold-level Metrics:


,model,fold,accuracy,precision,recall,f1,roc_auc,pr_auc,early_stop_epoch
0,TextCNN,1,0.988680,0.986522,0.992922,0.989712,0.999245,0.999371,5
1,TextCNN,2,0.990136,0.986273,0.995871,0.991049,0.999361,0.999448,7
2,TextCNN,3,0.993370,0.994392,0.993512,0.993952,0.999600,0.999656,5
3,TextCNN,4,0.987872,0.984512,0.993512,0.988992,0.999154,0.999246,6
4,TextCNN,5,0.991268,0.993201,0.990858,0.992028,0.999140,0.998486,5
5,TextLSTM,1,0.981727,0.985774,0.980832,0.983296,0.998159,0.998485,10
6,TextLSTM,2,0.983668,0.979872,0.990563,0.985188,0.998286,0.998301,6
7,TextLSTM,3,0.983668,0.988711,0.981421,0.985053,0.997910,0.998415,7
8,TextLSTM,4,0.980595,0.983161,0.981421,0.982290,0.997108,0.996386,9
9,TextLSTM,5,0.985446,0.986730,0.986730,0.986730,0.997868,0.998203,8



Word2Vec Regime: strict
Vocab Size: 50048


In [ ]:
# Running pipeline
source_dataset = 'Fake_News_Classification'

result_bundle, cross_dataset_results = new_train_loop_word2vec(
    dataset_name=source_dataset,
    datasets_map=DATASETS,
    vector_size=300,
    window=5,
    min_count=2,
    w2v_epochs=10,
    max_len=200,
    test_size=0.2,
    val_size=0.2,
    dl_epochs=10,
    dl_batch_size=32,
    dl_lr=1e-3,
    dl_patience=3,
    seed=42,
    transductive_w2v=False,
    enable_cv=True,
    cv_folds=5
)

print('\nRanking criterion: holdout test F1 (primary), CV metrics as diagnostics.')

print('\nML Ranked Results (holdout-F1 primary):')
display(result_bundle['ml_results'])

if result_bundle['ml_results_cv'] is not None:
    print('\nML CV Summary:')
    display(result_bundle['ml_results_cv'].sort_values('cv_f1_mean', ascending=False))

if result_bundle['ml_results_test'] is not None:
    print('\nML Holdout Test Metrics:')
    display(result_bundle['ml_results_test'].sort_values('f1', ascending=False))

if result_bundle['ml_fold_results'] is not None:
    print('\nML Fold-level Metrics:')
    display(result_bundle['ml_fold_results'])

print('\nDL Ranked Results (holdout-F1 primary):')
display(result_bundle['dl_results'])

if result_bundle['dl_results_cv'] is not None:
    print('\nDL CV Summary:')
    display(result_bundle['dl_results_cv'].sort_values('cv_f1_mean', ascending=False))

if result_bundle['dl_results_test'] is not None:
    print('\nDL Holdout Test Metrics:')
    display(result_bundle['dl_results_test'].sort_values('f1', ascending=False))

if result_bundle['dl_fold_results'] is not None:
    print('\nDL Fold-level Metrics:')
    display(result_bundle['dl_fold_results'])

print('\nWord2Vec Regime:', result_bundle['word2vec_regime'])
print('Vocab Size:', len(result_bundle['vocab']))


Initialising Word2Vec experiment on: Fake_News_Classification
Split summary (stratified):
Train: 25971 | Val: 6493 | Test: 8116

Word2Vec regime: strict

ML results with 5-fold CV (ranked by holdout_f1):
                model  holdout_f1  holdout_accuracy  cv_f1_mean  cv_f1_std  \
0                 SVC    0.978226          0.976589    0.975594   0.002578   
1  LogisticRegression    0.972770          0.970552    0.967432   0.002312   
2             XGBoost    0.967353          0.964638    0.964132   0.003068   
3        RandomForest    0.948454          0.943938    0.944863   0.003289   
4                 KNN    0.939648          0.933588    0.936662   0.002613   
5          GaussianNB    0.906116          0.900320    0.903646   0.003374   

   cv_roc_auc_mean  cv_pr_auc_mean  
0         0.993683        0.995050  
1         0.991610        0.992620  
2         0.991833        0.993504  
3         0.981943        0.985002  
4         0.971639        0.964677  
5         0.950582        

precision    recall        f1  \
model_type model              sub_row                                       
DL         TextCNN            Fake (0)       0.313539  0.353607  0.332370   
                              Real (1)       0.079643  0.067382  0.073001   
                              Weighted Avg   0.207446  0.223779  0.214723   
                              Overall        0.207446  0.223779  0.214723   
           TextCNNLSTM        Fake (0)       0.310240  0.367864  0.336603   
                              Real (1)       0.019003  0.014751  0.016609   
                              Weighted Avg   0.178138  0.207696  0.191457   
                              Overall        0.178138  0.207696  0.191457   
           TextLSTM           Fake (0)       0.314166  0.371716  0.340526   
                              Real (1)       0.028835  0.022472  0.025259   
                              Weighted Avg   0.184743  0.213303  0.197525   
                              Overall        0.184743  0.213303  0.197525   
ML         GaussianNB         Fake (0)       0.316308  0.322018  0.319137   
                              Real (1)       0.165121  0.161530  0.163306   
                              Weighted Avg   0.247731  0.249223  0.248454   
                              Overall        0.247731  0.249223  0.248454   
           KNN                Fake (0)       0.234311  0.222047  0.228014   
                              Real (1)       0.118433  0.125900  0.122052   
                              Weighted Avg   0.181750  0.178436  0.179951   
                              Overall        0.181750  0.178436  0.179951   
           LogisticRegression Fake (0)       0.221929  0.214803  0.218308   
                              Real (1)       0.089342  0.092798  0.091037   
                              Weighted Avg   0.161789  0.159463  0.160579   
                              Overall        0.161789  0.159463  0.160579   
           RandomForest       Fake (0)       0.241062  0.237597  0.239317   
                              Real (1)       0.097209  0.098892  0.098043   
                              Weighted Avg   0.175812  0.174682  0.175237   
                              Overall        0.175812  0.174682  0.175237   
           SVC                Fake (0)       0.227502  0.232021  0.229739   
                              Real (1)       0.052183  0.050935  0.051552   
                              Weighted Avg   0.147980  0.149882  0.148915   
                              Overall        0.147980  0.149882  0.148915   
           XGBoost            Fake (0)       0.221832  0.217764  0.219779   
                              Real (1)       0.078054  0.079778  0.078907   
                              Weighted Avg   0.156616  0.155175  0.155881   
                              Overall        0.156616  0.155175  0.155881   

                                            support  accuracy   roc_auc  \
model_type model              sub_row                                     
DL         TextCNN            Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.223779  0.095139   
           TextCNNLSTM        Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.207696  0.135991   
           TextLSTM           Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.213303  0.132687   
ML         GaussianNB         Fake (0)      34790.0       NaN      


Testing on unseen dataset: FakeNewsNet
Path: /content/drive/MyDrive/datasets/FakeNewsNet_processed.csv
Samples: 21844

Combined Unseen-Dataset Summary (ML + DL)


precision    recall        f1  \
model_type model              sub_row                                       
DL         TextCNN            Fake (0)       0.244994  0.944946  0.389106   
                              Real (1)       0.777525  0.061978  0.114805   
                              Weighted Avg   0.647781  0.277101  0.181635   
                              Overall        0.647781  0.277101  0.181635   
           TextCNNLSTM        Fake (0)       0.243647  0.999812  0.391812   
                              Real (1)       0.800000  0.000242  0.000484   
                              Weighted Avg   0.664452  0.243774  0.095826   
                              Overall        0.664452  0.243774  0.095826   
           TextLSTM           Fake (0)       0.244773  0.972379  0.391097   
                              Real (1)       0.790598  0.033592  0.064445   
                              Weighted Avg   0.657615  0.262315  0.144029   
                              Overall        0.657615  0.262315  0.144029   
ML         GaussianNB         Fake (0)       0.226018  0.278467  0.249516   
                              Real (1)       0.748806  0.692834  0.719733   
                              Weighted Avg   0.621436  0.591879  0.605171   
                              Overall        0.621436  0.591879  0.605171   
           KNN                Fake (0)       0.246558  0.824314  0.379580   
                              Real (1)       0.769193  0.188597  0.302921   
                              Weighted Avg   0.641860  0.343481  0.321598   
                              Overall        0.641860  0.343481  0.321598   
           LogisticRegression Fake (0)       0.247871  0.804021  0.378924   
                              Real (1)       0.772320  0.214139  0.335308   
                              Weighted Avg   0.644545  0.357856  0.345934   
                              Overall        0.644545  0.357856  0.345934   
           RandomForest       Fake (0)       0.246777  0.748027  0.371120   
                              Real (1)       0.765231  0.264556  0.393182   
                              Weighted Avg   0.638917  0.382348  0.387806   
                              Overall        0.638917  0.382348  0.387806   
           SVC                Fake (0)       0.244329  0.993799  0.392228   
                              Real (1)       0.832487  0.009926  0.019618   
                              Weighted Avg   0.689190  0.249634  0.110400   
                              Overall        0.689190  0.249634  0.110400   
           XGBoost            Fake (0)       0.244852  0.860203  0.381198   
                              Real (1)       0.763584  0.145442  0.244344   
                              Weighted Avg   0.637202  0.319584  0.277687   
                              Overall        0.637202  0.319584  0.277687   

                                            support  accuracy   roc_auc  \
model_type model              sub_row                                     
DL         TextCNN            Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.277101  0.495490   
           TextCNNLSTM        Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.243774  0.511271   
           TextLSTM           Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.262315  0.515702   
ML         GaussianNB         Fake (0)       5322.0       NaN      


Testing on unseen dataset: Fake_News_Detection
Path: /content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv
Samples: 38650

Combined Unseen-Dataset Summary (ML + DL)


...


DL Holdout Test Metrics:


,model,accuracy,precision,recall,f1,roc_auc,pr_auc,early_stop_epoch
0,TextCNN,0.988788,0.991980,0.987229,0.989599,0.999394,0.999534,5
1,TextCNNLSTM,0.984598,0.996735,0.974686,0.985587,0.993312,0.995991,6
2,TextLSTM,0.983489,0.996496,0.972862,0.984537,0.996809,0.997479,10



DL Fold-level Metrics:


,model,fold,accuracy,precision,recall,f1,roc_auc,pr_auc,early_stop_epoch
0,TextCNN,1,0.986139,0.988286,0.986032,0.987158,0.998793,0.998610,5
1,TextCNN,2,0.987525,0.990834,0.986032,0.988427,0.999204,0.999396,5
2,TextCNN,3,0.987217,0.988031,0.988312,0.988172,0.999246,0.999389,4
3,TextCNN,4,0.985831,0.991650,0.982036,0.986819,0.999129,0.999300,4
4,TextCNN,5,0.984596,0.993911,0.977474,0.985624,0.998879,0.999071,4
5,TextLSTM,1,0.978592,0.991538,0.968643,0.979957,0.997055,0.997704,10
6,TextLSTM,2,0.983829,0.992475,0.977480,0.984920,0.996057,0.997438,8
7,TextLSTM,3,0.982597,0.995620,0.972064,0.983701,0.993405,0.995791,9
8,TextLSTM,4,0.982135,0.988758,0.978044,0.983372,0.997362,0.997974,9
9,TextLSTM,5,0.981208,0.991292,0.973767,0.982451,0.996329,0.997098,10



Word2Vec Regime: strict
Vocab Size: 54220


In [22]:
# Running pipeline
source_dataset = 'ISOT'

result_bundle, cross_dataset_results = new_train_loop_word2vec(
    dataset_name=source_dataset,
    datasets_map=DATASETS,
    vector_size=300,
    window=5,
    min_count=2,
    w2v_epochs=10,
    max_len=200,
    test_size=0.2,
    val_size=0.2,
    dl_epochs=10,
    dl_batch_size=32,
    dl_lr=1e-3,
    dl_patience=3,
    seed=42,
    transductive_w2v=False,
    enable_cv=True,
    cv_folds=5
)

print('\nRanking criterion: holdout test F1 (primary), CV metrics as diagnostics.')

print('\nML Ranked Results (holdout-F1 primary):')
display(result_bundle['ml_results'])

if result_bundle['ml_results_cv'] is not None:
    print('\nML CV Summary:')
    display(result_bundle['ml_results_cv'].sort_values('cv_f1_mean', ascending=False))

if result_bundle['ml_results_test'] is not None:
    print('\nML Holdout Test Metrics:')
    display(result_bundle['ml_results_test'].sort_values('f1', ascending=False))

if result_bundle['ml_fold_results'] is not None:
    print('\nML Fold-level Metrics:')
    display(result_bundle['ml_fold_results'])

print('\nDL Ranked Results (holdout-F1 primary):')
display(result_bundle['dl_results'])

if result_bundle['dl_results_cv'] is not None:
    print('\nDL CV Summary:')
    display(result_bundle['dl_results_cv'].sort_values('cv_f1_mean', ascending=False))

if result_bundle['dl_results_test'] is not None:
    print('\nDL Holdout Test Metrics:')
    display(result_bundle['dl_results_test'].sort_values('f1', ascending=False))

if result_bundle['dl_fold_results'] is not None:
    print('\nDL Fold-level Metrics:')
    display(result_bundle['dl_fold_results'])

print('\nWord2Vec Regime:', result_bundle['word2vec_regime'])
print('Vocab Size:', len(result_bundle['vocab']))


Initialising Word2Vec experiment on: ISOT
Split summary (stratified):
Train: 25022 | Val: 6256 | Test: 7820

Word2Vec regime: strict

ML results with 5-fold CV (ranked by holdout_f1):
                model  holdout_f1  holdout_accuracy  cv_f1_mean  cv_f1_std  \
0                 SVC    0.987672          0.988747    0.988262   0.002783   
1  LogisticRegression    0.985337          0.986573    0.985590   0.002299   
2             XGBoost    0.972973          0.975448    0.974429   0.001992   
3        RandomForest    0.951327          0.956138    0.952426   0.002894   
4                 KNN    0.942828          0.948977    0.945296   0.002356   
5          GaussianNB    0.907080          0.914066    0.900350   0.004232   

   cv_roc_auc_mean  cv_pr_auc_mean  
0         0.998890        0.998871  
1         0.998577        0.998640  
2         0.997267        0.997146  
3         0.991462        0.990795  
4         0.982432        0.977159  
5         0.960785        0.941405  

TextCNN:

precision    recall        f1  \
model_type model              sub_row                                       
DL         TextCNN            Fake (0)       0.990942  0.613165  0.757569   
                              Real (1)       0.680659  0.993248  0.807766   
                              Weighted Avg   0.850201  0.785566  0.780338   
                              Overall        0.850201  0.785566  0.780338   
           TextCNNLSTM        Fake (0)       0.987611  0.618684  0.760780   
                              Real (1)       0.683208  0.990651  0.808695   
                              Weighted Avg   0.849537  0.787404  0.782514   
                              Overall        0.849537  0.787404  0.782514   
           TextLSTM           Fake (0)       0.984648  0.637884  0.774212   
                              Real (1)       0.693718  0.988019  0.815117   
                              Weighted Avg   0.852685  0.796702  0.792766   
                              Overall        0.852685  0.796702  0.792766   
ML         GaussianNB         Fake (0)       0.833339  0.677235  0.747221   
                              Real (1)       0.682770  0.836842  0.751995   
                              Weighted Avg   0.765043  0.749631  0.749387   
                              Overall        0.765043  0.749631  0.749387   
           KNN                Fake (0)       0.896470  0.774562  0.831069   
                              Real (1)       0.766654  0.892244  0.824695   
                              Weighted Avg   0.837587  0.827941  0.828178   
                              Overall        0.837587  0.827941  0.828178   
           LogisticRegression Fake (0)       0.937884  0.758638  0.838792   
                              Real (1)       0.763658  0.939474  0.842491   
                              Weighted Avg   0.858857  0.840663  0.840470   
                              Overall        0.858857  0.840663  0.840470   
           RandomForest       Fake (0)       0.905885  0.760563  0.826888   
                              Real (1)       0.758277  0.904813  0.825090   
                              Weighted Avg   0.838932  0.825993  0.826072   
                              Overall        0.838932  0.825993  0.826072   
           SVC                Fake (0)       0.948822  0.756194  0.841627   
                              Real (1)       0.764015  0.950866  0.847261   
                              Weighted Avg   0.864996  0.844495  0.844183   
                              Overall        0.864996  0.844495  0.844183   
           XGBoost            Fake (0)       0.935071  0.762087  0.839763   
                              Real (1)       0.765630  0.936253  0.842389   
                              Weighted Avg   0.858215  0.841087  0.840954   
                              Overall        0.858215  0.841087  0.840954   

                                            support  accuracy   roc_auc  \
model_type model              sub_row                                     
DL         TextCNN            Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.785566  0.920439   
           TextCNNLSTM        Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.787404  0.891011   
           TextLSTM           Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.796702  0.866520   
ML         GaussianNB         Fake (0)      34790.0       NaN      


Testing on unseen dataset: FakeNewsNet
Path: /content/drive/MyDrive/datasets/FakeNewsNet_processed.csv
Samples: 21844

Combined Unseen-Dataset Summary (ML + DL)


precision    recall        f1  \
model_type model              sub_row                                       
DL         TextCNN            Fake (0)       0.237500  0.003570  0.007034   
                              Real (1)       0.756341  0.996308  0.859897   
                              Weighted Avg   0.629932  0.754441  0.652108   
                              Overall        0.629932  0.754441  0.652108   
           TextCNNLSTM        Fake (0)       0.271186  0.018038  0.033827   
                              Real (1)       0.756817  0.984384  0.855730   
                              Weighted Avg   0.638500  0.748947  0.655484   
                              Overall        0.638500  0.748947  0.655484   
           TextLSTM           Fake (0)       0.190909  0.003946  0.007732   
                              Real (1)       0.756096  0.994613  0.859107   
                              Weighted Avg   0.618396  0.753250  0.651681   
                              Overall        0.618396  0.753250  0.651681   
ML         GaussianNB         Fake (0)       0.258547  0.727546  0.381515   
                              Real (1)       0.788876  0.327926  0.463275   
                              Weighted Avg   0.659668  0.425288  0.443355   
                              Overall        0.659668  0.425288  0.443355   
           KNN                Fake (0)       0.241388  0.173807  0.202097   
                              Real (1)       0.755885  0.824053  0.788498   
                              Weighted Avg   0.630535  0.665629  0.645630   
                              Overall        0.630535  0.665629  0.645630   
           LogisticRegression Fake (0)       0.232039  0.187523  0.207420   
                              Real (1)       0.753520  0.800085  0.776105   
                              Weighted Avg   0.626468  0.650842  0.637552   
                              Overall        0.626468  0.650842  0.637552   
           RandomForest       Fake (0)       0.239166  0.137918  0.174949   
                              Real (1)       0.755632  0.858673  0.803864   
                              Weighted Avg   0.629802  0.683071  0.650638   
                              Overall        0.629802  0.683071  0.650638   
           SVC                Fake (0)       0.181818  0.001127  0.002241   
                              Real (1)       0.756270  0.998366  0.860616   
                              Weighted Avg   0.616312  0.755402  0.651484   
                              Overall        0.616312  0.755402  0.651484   
           XGBoost            Fake (0)       0.226148  0.096204  0.134985   
                              Real (1)       0.754341  0.893960  0.818237   
                              Weighted Avg   0.625654  0.699597  0.651772   
                              Overall        0.625654  0.699597  0.651772   

                                            support  accuracy   roc_auc  \
model_type model              sub_row                                     
DL         TextCNN            Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.754441  0.507910   
           TextCNNLSTM        Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.748947  0.525804   
           TextLSTM           Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.753250  0.474499   
ML         GaussianNB         Fake (0)       5322.0       NaN      


Testing on unseen dataset: Fake_News_Detection
Path: /content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv
Samples: 38650

Combined Unseen-Dataset Summary (ML + DL)


...


DL Holdout Test Metrics:


,model,accuracy,precision,recall,f1,roc_auc,pr_auc,early_stop_epoch
0,TextCNNLSTM,0.998465,0.997768,0.998883,0.998325,0.999920,0.999909,7
1,TextCNN,0.997826,0.997487,0.997766,0.997627,0.999985,0.999983,5
2,TextLSTM,0.997570,0.996654,0.998045,0.997349,0.999625,0.999484,10



DL Fold-level Metrics:


,model,fold,accuracy,precision,recall,f1,roc_auc,pr_auc,early_stop_epoch
0,TextCNN,1,0.997922,0.998950,0.996508,0.997728,0.999948,0.999939,4
1,TextCNN,2,0.998082,0.997905,0.997905,0.997905,0.999967,0.999962,5
2,TextCNN,3,0.999201,0.998953,0.999302,0.999128,0.999997,0.999996,9
3,TextCNN,4,0.998561,0.997560,0.999302,0.998430,0.999957,0.999948,5
4,TextCNN,5,0.998561,0.997560,0.999302,0.998430,0.999974,0.999969,5
5,TextLSTM,1,0.989290,0.986097,0.990573,0.988330,0.997549,0.997343,10
6,TextLSTM,2,0.999041,0.998604,0.999302,0.998953,0.999995,0.999994,10
7,TextLSTM,3,0.998242,0.997906,0.998255,0.998081,0.999960,0.999951,9
8,TextLSTM,4,0.994404,0.995100,0.992668,0.993882,0.999614,0.999514,10
9,TextLSTM,5,0.997122,0.996511,0.997207,0.996859,0.999721,0.999694,10



Word2Vec Regime: strict
Vocab Size: 50172
